# CS-781 Phase 4 — Court Judgment Prediction (CJPE)

Nithin T M · 230709 · `n_230709` · window 2026-10-17 → 2026-10-21 23:59 IST

**Two runtimes, same notebook.** Open it twice (File → Open in a second tab).

| runtime | run cells | what it does | cost |
|---|---|---|---|
| **A: CPU** (standard) | 1, 2, 3, then **A1–A4** | data gate, integrity check vs Phase-1 corpus, TF-IDF zips | ~0 CU |
| **B: GPU L4** (never T4) | 1, 2, 3, then **B1**, B2 to watch | tokenise, bench, 8 runs + KD, ensembles → zips | ~4 GPU-h |

Both write to `MyDrive/cs781_capstone/phase4/`. Everything is resumable: re-running B1 skips finished runs.
Keep the B tab open while it runs. Zips land in `phase4/submissions/`.

In [ ]:
# 1. setup (both runtimes) -- mount, paths, HF token, Phase-1 file check
import os, sys, json, glob, subprocess, time
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')          # never rmtree the mountpoint; force_remount=True if stale
BASE = '/content/drive/MyDrive/cs781_capstone'
P1, P4 = f'{BASE}/phase1', f'{BASE}/phase4'
CODE = f'{P4}/code'
for d in (CODE, f'{P4}/logs', f'{P4}/results', f'{P4}/submissions'):
    os.makedirs(d, exist_ok=True)
tok_file = f'{BASE}/.hf_token'
if not os.path.exists(tok_file):
    raise SystemExit(f'put your HF read token in {tok_file} (one line) and re-run')
os.environ['HF_TOKEN'] = open(tok_file).read().strip()
os.environ.update(CS781_BASE=BASE, CS781_P1=P1, CS781_P4=P4, PYTHONUNBUFFERED='1')
need = [f'{P1}/shared/tokenizer/bpe_legal_32k.json'] + \
       [f'{P1}/shared/lib/{f}' for f in ('cs781_bpe.py', 'cs781_model.py', 'cs781_model_ed.py', 'cs781_model_ptr.py')] + \
       [f'{P1}/{m}/ckpt/best.pt' for m in ('M1_SC', 'M2_LexArcED', 'M3_LexArcL', 'M4_LexArcP')]
miss = [p for p in need if not os.path.exists(p)]
print('missing:', miss) if miss else print('Phase-1 files OK')
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip() or 'CPU runtime')
except FileNotFoundError:
    print('CPU runtime')

## 2. Code files
Each cell below writes one file to `phase4/code/` on Drive. Run them all (both runtimes; rewriting the same
content is harmless). Source of truth: `capstone_Project/phase4/code/` in the repo.

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/phase4_data.py
#!/usr/bin/env python3
"""CS-781 Phase 4 (CJPE) -- data, token cache, metrics, thresholds, submissions.

    python phase4_data.py gate            # stats + leakage checks (CPU, no tokenizer needed)
    python phase4_data.py tokenize        # BPE token cache for the GPU runs (CPU, all cores)

Everything else imports from here. Paths come from the environment so the same
code runs on Colab (Drive) and locally:

    CS781_BASE   default /content/drive/MyDrive/cs781_capstone
    CS781_P4     where results/, submissions/, cache/ and logs/ go
    CS781_P4_DATA  optional directory holding train.json / dev.json / test.json

Run-file format shared by every model (GPU runs, lexical, ensembles), so the
ensembler never needs to know what produced a file:

    results/<tag>/dev.pt   {"ids": [...], "z": float32 tensor}   z = logit(P(label=1))
    results/<tag>/test.pt  same, test ids
    results/<tag>/meta.json  dev metrics, threshold, params, cost
"""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import os
import re
import sys
import time
import zipfile

BASE = os.environ.get("CS781_BASE", "/content/drive/MyDrive/cs781_capstone")
P1 = os.environ.get("CS781_P1", f"{BASE}/phase1")
P4 = os.environ.get("CS781_P4", f"{BASE}/phase4")
LOCAL = os.environ.get("CS781_P4_DATA")
REPO = "Exploration-Lab/CS-781-Capstone"
SUB = "Phase_4_CJPE"
FILES = {"train": "Train/train.json", "dev": "Train/dev.json", "test": "Test/test.json"}


def log(*a):
    print(*a, flush=True)


# --------------------------------------------------------------------- load --
def fetch(split):
    name = FILES[split]
    for root in [LOCAL, f"{P4}/data"]:
        if root:
            for p in (os.path.join(root, name), os.path.join(root, os.path.basename(name))):
                if os.path.exists(p):
                    return p
    from huggingface_hub import hf_hub_download
    return hf_hub_download(REPO, f"{SUB}/{name}", repo_type="dataset")


def read_records(path):
    """A JSON list, a dict id -> record, or JSON lines."""
    with open(path, encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        if head == "[" or head == "{":
            try:
                raw = json.load(f)
            except json.JSONDecodeError:
                f.seek(0)
                raw = [json.loads(l) for l in f if l.strip()]
        else:
            raw = [json.loads(l) for l in f if l.strip()]
    if isinstance(raw, dict):
        if "id" in raw and "text" in raw:
            raw = [raw]
        else:
            raw = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v}
                   for k, v in raw.items()]
    return raw


def as_label(v):
    if isinstance(v, bool):
        return int(v)
    if isinstance(v, (int, float)):
        return int(v)
    s = str(v).strip().lower()
    return {"0": 0, "1": 1, "rejected": 0, "accepted": 1}[s]


def load_split(split):
    """[(id, text, label|None)] in file order, one record per id. Test labels are never read.

    The files merge ILDC's single- and multi-appeal sets, so the same case id can
    appear twice. Repeats are dropped (first copy kept) and counted: same text,
    different text, and -- for labelled splits -- different label.
    """
    out, seen = [], {}
    dup = {"same_text": 0, "diff_text": 0, "diff_label": 0}
    for r in read_records(fetch(split)):
        did = str(r["id"])
        text = r["text"]
        if isinstance(text, list):
            text = "\n".join(map(str, text))
        text = str(text)
        lab = None
        if split != "test":
            lab = as_label(r["label"])
        if did in seen:
            t0, l0 = seen[did]
            dup["same_text" if t0 == text else "diff_text"] += 1
            if split != "test" and l0 != lab:
                dup["diff_label"] += 1
            continue
        seen[did] = (text, lab)
        out.append((did, text, lab))
    if any(dup.values()):
        log(f"  {split}: {sum(dup.values()) - dup['diff_label']} repeated ids dropped {dup} -> {len(out)} unique")
    return out


def text_key(t):
    return hashlib.sha1(norm(t).encode()).hexdigest()


def test_copies():
    """{"train": [ids], "dev": [ids]}: labelled documents whose text IS a test document.

    The released dev.json holds ILDC's 994 dev cases AND its 1,517 test cases with
    their labels (2,511 = 994 + 1,517; every test text has an exact copy there).
    Selecting checkpoints or thresholds on those copies would be tuning on the
    test labels, so they are removed from every labelled split before use.
    Cached in cache/test_copies.json after the first call.
    """
    cp = f"{P4}/cache/test_copies.json"
    if os.path.exists(cp):
        return json.load(open(cp))
    data = {s: load_split(s) for s in FILES}
    tk = {text_key(t) for _, t, _ in data["test"]}
    out = {s: sorted(d for d, t, _ in data[s] if text_key(t) in tk) for s in ("train", "dev")}
    os.makedirs(os.path.dirname(cp), exist_ok=True)
    atomic_json(out, cp)
    log(f"  test copies inside labelled data: train {len(out['train'])}, dev {len(out['dev'])} -> {cp}")
    return out


def load_all(splits=("train", "dev", "test"), clean=True):
    """clean=True drops labelled documents that are copies of test documents (see test_copies)."""
    data = {s: load_split(s) for s in splits}
    if clean:
        drop = test_copies()
        for s in ("train", "dev"):
            if s in data and drop.get(s):
                bad = set(drop[s])
                n0 = len(data[s])
                data[s] = [r for r in data[s] if r[0] not in bad]
                log(f"  {s}: removed {n0 - len(data[s])} copies of test documents -> {len(data[s])}")
    return data


def drop_test_copies(c, split):
    """Same filter for a token cache dict."""
    bad = set(test_copies().get(split, []))
    if not bad:
        return c
    keep = [i for i, d in enumerate(c["ids"]) if d not in bad]
    log(f"  {split}: removed {len(c['ids']) - len(keep)} copies of test documents -> {len(keep)}")
    return dict(ids=[c["ids"][i] for i in keep], tok=[c["tok"][i] for i in keep],
                label=[c["label"][i] for i in keep])


def load_jsonl(path, limit=None):
    """External labelled docs, one {"id","text","label"} per line -> [(id, text, label)]."""
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                r = json.loads(line)
                out.append((str(r["id"]), r["text"], as_label(r["label"])))
                if limit and len(out) >= limit:
                    break
    return out


# -------------------------------------------------------------- token cache --
def _tok_worker(args):
    path, texts = args
    sys.path.insert(0, f"{P1}/shared/lib")
    from cs781_bpe import BPETokenizer
    import numpy as np
    tok = BPETokenizer.load(path)
    return [np.asarray(tok.encode(t), dtype=np.uint16) for t in texts]


def ext_cache(path, procs=None):
    """Token cache for an external jsonl (cached under its basename)."""
    name = "ext_" + os.path.splitext(os.path.basename(path))[0]
    return token_cache(name, recs=None if os.path.exists(f"{P4}/cache/tok_{name}.pt") else load_jsonl(path),
                       procs=procs)


def token_cache(split, recs=None, procs=None):
    """{"ids": [...], "tok": [np.uint16 arrays], "label": [...]}, cached on Drive.

    Whole documents are tokenised once (the BPE is pure Python); every view
    (tail, head+tail, spread chunks) is cut from these arrays at train time.
    """
    import numpy as np
    import torch
    d = f"{P4}/cache"
    os.makedirs(d, exist_ok=True)
    cp = f"{d}/tok_{split}.pt"
    if os.path.exists(cp):
        try:
            return torch.load(cp, weights_only=False)
        except Exception as e:                      # truncated by a killed run
            log(f"  token cache {cp} unreadable ({e}); rebuilding")
    recs = recs if recs is not None else load_split(split)
    path = f"{P1}/shared/tokenizer/bpe_legal_32k.json"
    procs = procs or max(1, os.cpu_count() or 1)
    t0 = time.time()
    texts = [" " + t.strip() for _, t, _ in recs]
    # contiguous shards keep each worker's chunk cache warm on similar text
    k = max(1, math.ceil(len(texts) / (procs * 8)))
    shards = [(path, texts[i:i + k]) for i in range(0, len(texts), k)]
    if procs > 1:
        import multiprocessing as mp
        with mp.get_context("spawn" if os.name == "nt" else "fork").Pool(procs) as pool:
            parts = pool.map(_tok_worker, shards)
    else:
        parts = [_tok_worker(s) for s in shards]
    toks = [a for p in parts for a in p]
    n = sum(len(a) for a in toks)
    out = dict(ids=[r[0] for r in recs], tok=toks, label=[r[2] for r in recs])
    tmp = f"{cp}.tmp{os.getpid()}"
    torch.save(out, tmp)
    os.replace(tmp, cp)
    log(f"  tokenised {split}: {len(toks)} docs, {n/1e6:.1f}M tokens in {time.time()-t0:.0f}s "
        f"({n/max(1e-6, time.time()-t0)/1e3:.0f}k tok/s, {procs} procs) -> {cp}")
    return out


# ------------------------------------------------------------------ metrics --
def macro_f1(y, p):
    """sklearn f1_score(average='macro', labels=[0,1], zero_division=0)."""
    tp = sum(1 for a, b in zip(y, p) if a == 1 and b == 1)
    tn = sum(1 for a, b in zip(y, p) if a == 0 and b == 0)
    fp = sum(1 for a, b in zip(y, p) if a == 0 and b == 1)
    fn = sum(1 for a, b in zip(y, p) if a == 1 and b == 0)
    f1 = lambda t, f_p, f_n: 2 * t / (2 * t + f_p + f_n) if t else 0.0
    return (f1(tp, fp, fn) + f1(tn, fn, fp)) / 2


def metrics(y, p):
    tp = sum(1 for a, b in zip(y, p) if a == 1 and b == 1)
    fp = sum(1 for a, b in zip(y, p) if a == 0 and b == 1)
    fn = sum(1 for a, b in zip(y, p) if a == 1 and b == 0)
    return dict(macro_f1=macro_f1(y, p), precision=tp / (tp + fp) if tp + fp else 0.0,
                recall=tp / (tp + fn) if tp + fn else 0.0,
                acc=sum(a == b for a, b in zip(y, p)) / max(1, len(y)),
                pos_rate=sum(p) / max(1, len(p)))


def best_threshold(y, z):
    """Threshold on z maximising dev Macro-F1. Ties -> the one closest to 0."""
    order = sorted(range(len(z)), key=lambda i: -z[i])
    P = sum(y)
    N = len(y) - P
    best = (macro_f1(y, [1 if v > 0 else 0 for v in z]), 0.0)
    tp = fp = 0
    for r, i in enumerate(order):
        tp += y[i]
        fp += 1 - y[i]
        fn, tn = P - tp, N - fp
        f = ((2 * tp / (2 * tp + fp + fn) if tp else 0) + (2 * tn / (2 * tn + fn + fp) if tn else 0)) / 2
        nxt = z[order[r + 1]] if r + 1 < len(order) else z[i] - 1.0
        thr = (z[i] + nxt) / 2
        if f > best[0] + 1e-12 or (abs(f - best[0]) < 1e-12 and abs(thr) < abs(best[1])):
            best = (f, thr)
    return best[1], best[0]


def rate_threshold(z_test, rate):
    """Threshold giving the test set the same positive rate as `rate` (dev's)."""
    s = sorted(z_test, reverse=True)
    k = int(round(rate * len(s)))
    if k <= 0:
        return s[0] + 1.0
    if k >= len(s):
        return s[-1] - 1.0
    return (s[k - 1] + s[k]) / 2


def auc(y, z):
    pos = [v for v, t in zip(z, y) if t == 1]
    neg = [v for v, t in zip(z, y) if t == 0]
    if not pos or not neg:
        return float("nan")
    allv = sorted((v, t) for v, t in zip(z, y))
    rank, i, rs = 0, 0, 0.0
    while i < len(allv):
        j = i
        while j < len(allv) and allv[j][0] == allv[i][0]:
            j += 1
        avg = (i + j + 1) / 2
        rs += avg * sum(1 for k in range(i, j) if allv[k][1] == 1)
        i = j
    return (rs - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def summarise(y, z, thr=None):
    t, fbest = best_threshold(y, z)
    thr = t if thr is None else thr
    m05 = metrics(y, [1 if v > 0 else 0 for v in z])
    mt = metrics(y, [1 if v > thr else 0 for v in z])
    return dict(f1_at0=m05["macro_f1"], f1_tuned=mt["macro_f1"], thr=thr, auc=auc(y, z),
                precision=mt["precision"], recall=mt["recall"], acc=mt["acc"],
                pos_rate=mt["pos_rate"])


# -------------------------------------------------------------- run files ---
def save_run(tag, dev_ids, dev_z, test_ids, test_z, meta, extra=None):
    import torch
    rd = f"{P4}/results/{tag}"
    os.makedirs(rd, exist_ok=True)
    torch.save(dict(ids=list(dev_ids), z=torch.as_tensor(dev_z, dtype=torch.float32)), f"{rd}/dev.pt")
    torch.save(dict(ids=list(test_ids), z=torch.as_tensor(test_z, dtype=torch.float32)), f"{rd}/test.pt")
    for k, v in (extra or {}).items():
        torch.save(v, f"{rd}/{k}.pt")
    atomic_json(meta, f"{rd}/meta.json", indent=1)
    return rd


def load_run(tag):
    import torch
    rd = f"{P4}/results/{tag}"
    d = torch.load(f"{rd}/dev.pt", weights_only=False)
    t = torch.load(f"{rd}/test.pt", weights_only=False)
    meta = json.load(open(f"{rd}/meta.json"))
    return d, t, meta


def atomic_json(obj, path, **kw):
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, **kw)
    os.replace(tmp, path)


# --------------------------------------------------------------- submission --
def validate_prediction(pred, test_ids):
    """Every rule evaluate_phase4.py rejects on."""
    assert isinstance(pred, list) and pred, "prediction must be a non-empty JSON list"
    seen = set()
    for n, r in enumerate(pred):
        assert isinstance(r, dict) and "id" in r and "pred" in r, f"item {n} needs id and pred"
        assert isinstance(r["id"], str) and r["id"].strip(), f"item {n}: bad id"
        assert not isinstance(r["pred"], bool) and r["pred"] in (0, 1) and isinstance(r["pred"], int), \
            f"item {n}: pred must be int 0/1"
        assert r["id"] not in seen, f"duplicate id {r['id']}"
        seen.add(r["id"])
    want = set(test_ids)
    assert seen == want, f"id mismatch: missing {len(want - seen)}, extra {len(seen - want)}"
    return True


def write_submission(tag, test_ids, pred, dev=None, note="", kind="model"):
    """pred: {id: 0/1}. Writes submissions/<tag>.zip and appends to the tracker.

    kind: "model" (only Phase-1-derived models) or "hybrid" (anything else in
    the mix, e.g. TF-IDF). The report must keep the two apart.
    """
    out = [{"id": d, "pred": int(pred[d])} for d in test_ids]
    validate_prediction(out, test_ids)
    sd = f"{P4}/submissions"
    os.makedirs(sd, exist_ok=True)
    pj = f"{sd}/prediction_{tag}.json"
    with open(pj, "w", encoding="utf-8") as f:
        json.dump(out, f)
    zp = f"{sd}/{tag}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, "prediction.json")
    with zipfile.ZipFile(zp) as z:                       # re-read what we ship
        assert z.namelist() == ["prediction.json"], z.namelist()
        validate_prediction(json.loads(z.read("prediction.json")), test_ids)
    h = hashlib.sha1(json.dumps(out, sort_keys=True).encode()).hexdigest()[:12]
    tr_path = f"{sd}/tracker.json"
    tr = json.load(open(tr_path)) if os.path.exists(tr_path) else []
    dup = [t["tag"] for t in tr if t.get("sha") == h and t["tag"] != tag]
    tr = [t for t in tr if t["tag"] != tag]
    tr.append(dict(tag=tag, zip=os.path.basename(zp), sha=h, kind=kind,
                   dev=dev, pos_rate=sum(out[i]["pred"] for i in range(len(out))) / len(out),
                   note=note, test=None, time=time.strftime("%Y-%m-%d %H:%M")))
    atomic_json(tr, tr_path, indent=1)
    pr = sum(r["pred"] for r in out) / len(out)
    log(f"  WROTE {zp}  sha {h}  test pos-rate {pr:.3f}"
        + (f"  -- IDENTICAL to {dup}, will not count" if dup else ""))
    return zp


# --------------------------------------------------------------------- gate --
WORD = re.compile(r"[a-z0-9]+")

OUTCOME = [  # phrases that state the outcome outright; ILDC is supposed to strip them
    r"appeals? (?:is|are|stands?|shall stand) (?:hereby )?(?:allowed|dismissed|accepted|rejected|disposed)",
    r"petitions? (?:is|are|stands?) (?:hereby )?(?:allowed|dismissed|accepted|rejected|disposed)",
    r"(?:we|i) (?:therefore |accordingly )?(?:allow|dismiss) (?:the|this|these) (?:appeal|petition)",
    r"in the result,? the appeal",
    r"leave (?:is|was) granted",
]


def norm(t):
    return " ".join(WORD.findall(t.lower()))


def gate(a):
    data = load_all(clean=False)
    for s, recs in data.items():
        n = len(recs)
        lens = sorted(len(t) for _, t, _ in recs)
        wl = sorted(len(t.split()) for _, t, _ in recs)
        q = lambda v, p: v[min(len(v) - 1, int(p * len(v)))]
        lab = collections.Counter(l for _, _, l in recs)
        log(f"{s:5s}: {n:6d} docs | words median {q(wl,.5)} p90 {q(wl,.9)} max {wl[-1]} | "
            f"chars median {q(lens,.5)} | labels {dict(lab)}")
    # ids: ILDC-style ids tell us the source
    for s, recs in data.items():
        log(f"  {s} id examples: {[r[0] for r in recs[:4]]}")

    # outcome phrases in the last 1500 chars
    pats = [re.compile(p, re.I) for p in OUTCOME]
    for s, recs in data.items():
        hit = collections.Counter()
        by_lab = collections.defaultdict(collections.Counter)
        for _, t, l in recs:
            tail = t[-1500:]
            for k, p in enumerate(pats):
                if p.search(tail):
                    hit[k] += 1
                    by_lab[k][l] += 1
        log(f"  {s}: outcome phrases in last 1500 chars: " +
            ", ".join(f"p{k}={hit[k]} {dict(by_lab[k])}" for k in range(len(pats))))
    log("\n  last 300 chars of 3 train docs per label:")
    shown = collections.Counter()
    for _, t, l in data["train"]:
        if shown[l] < 3:
            shown[l] += 1
            log(f"   [label {l}] ...{t[-300:]!r}")

    # exact and 13-gram overlap between test and train/dev
    def key(t):
        return hashlib.sha1(norm(t).encode()).hexdigest()

    lab_keys = {key(t): (s, d, l) for s in ("train", "dev") for d, t, l in data[s]}
    ex = [(d, lab_keys[key(t)]) for d, t, _ in data["test"] if key(t) in lab_keys]
    log(f"\n  exact (normalised) duplicates test->train/dev: {len(ex)} of {len(data['test'])}")
    dd = [(d, lab_keys[key(t)]) for d, t, _ in data["dev"] if key(t) in lab_keys and lab_keys[key(t)][0] == "train"]
    log(f"  exact duplicates dev->train: {len(dd)} of {len(data['dev'])}")
    tr_keys = collections.defaultdict(set)
    for d, t, l in data["train"]:
        tr_keys[key(t)].add(l)
    conflict = sum(1 for v in tr_keys.values() if len(v) > 1)
    dup_train = len(data["train"]) - len(tr_keys)
    log(f"  duplicate texts inside train: {dup_train}; with conflicting labels: {conflict}")

    # 13-gram near-duplicates: sample shingles of each test doc's tail
    n = 13
    idx = collections.defaultdict(list)
    for s in ("train", "dev"):
        for j, (d, t, l) in enumerate(data[s]):
            w = norm(t[-6000:]).split()
            for i in range(0, max(0, len(w) - n + 1), 25):
                idx[hash(" ".join(w[i:i + n]))].append((s, j))
    near = 0
    for d, t, _ in data["test"]:
        w = norm(t[-6000:]).split()
        c = collections.Counter()
        for i in range(max(0, len(w) - n + 1)):
            for h in idx.get(hash(" ".join(w[i:i + n])), ()):
                c[h] += 1
        if c and c.most_common(1)[0][1] >= 20:
            near += 1
    log(f"  test docs whose tail shares >=20 sampled 13-grams with one train/dev doc: {near}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["gate", "tokenize"])
    ap.add_argument("--procs", type=int, default=None)
    a = ap.parse_args()
    if a.cmd == "gate":
        gate(a)
    else:
        for s in ("dev", "test", "train"):
            c = token_cache(s, procs=a.procs)
            L = sorted(len(x) for x in c["tok"])
            q = lambda p: L[min(len(L) - 1, int(p * len(L)))]
            log(f"  {s}: BPE tokens/doc median {q(.5)} p75 {q(.75)} p90 {q(.9)} max {L[-1]} "
                f"| >512 {sum(x > 512 for x in L)/len(L):.2f} >2048 {sum(x > 2048 for x in L)/len(L):.2f} "
                f">4096 {sum(x > 4096 for x in L)/len(L):.2f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/phase4_peft.py
"""CS-781 Phase 4 -- adaptation methods for the Phase-1 trunks (Phase-3 module + four more).

Every method here is implemented directly on the Phase-1 modules (no peft
library: the trunks are custom nn.Modules, and prefix/prompt tuning in peft only
works on HF transformers classes).

    method    trainable in the trunk                         reference
    head      nothing (frozen trunk, train the RR head)      floor
    full      everything                                     ceiling to compare against
    lora      B A added to frozen W, scale alpha/r           Hu et al. 2021
    rslora    same, scale alpha/sqrt(r), higher rank         Kalajdzievski 2023
    qlora     frozen W stored as NF4 (+ double-quantised     Dettmers et al. 2023
              absmax), LoRA on every linear layer
    adalora   W + P diag(E) Q, rank budget reallocated by    Zhang et al. 2023
              sensitivity-based importance during training
    lorma     (I + alpha/r B A) W -- multiplicative          Bihany, Patel & Modi 2025
    ia3       learned rescaling of k, v and FFN activation   Liu et al. 2022
    prompt    P soft tokens prepended to the input           Lester et al. 2021
    prefix    P learned states at every layer (MLP reparam)  Li & Liang 2021
    norm      RMSNorm gains only                             (Phase 3)
    bitfit    bias vectors only. The Phase-1 linears have    Zaken et al. 2022
              no biases, so every one gets a zero-initialised
              bias and those are the ONLY trunk parameters
              trained -- BitFit on a bias-free network.
    adapter   Houlsby bottleneck (d -> m -> d, GELU, up-proj  Houlsby et al. 2019
              zero-init) after the attention output and after
              the FFN of every block, before the residual add
    partial   top-k blocks (+ final norm) fully trainable,   layer-wise FT
              the rest frozen; optional layer-wise LR decay
    unfreeze  head first, then one more block from the top    Howard & Ruder 2018
              every `unfreeze_every` steps (gradual           (ULMFiT)
              unfreezing), discriminative LRs per depth

`prompt` and `prefix` are handled in the model's forward pass (they change the
sequence, not a weight); this module only builds their parameters.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------- LoRA ------
class LoRALinear(nn.Module):
    """y = W x + s * B(A x);  s = alpha/r (LoRA) or alpha/sqrt(r) (rsLoRA).

    rsLoRA's point: with alpha/r the update's scale shrinks as r grows, so a
    larger rank learns no faster and the extra capacity goes unused. alpha/sqrt(r)
    keeps the learning dynamics rank-stable, which is what makes r=64 worth
    trying at all.
    """

    def __init__(self, base, r, alpha=16.0, rs=False, dropout=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / (math.sqrt(r) if rs else r)
        self.drop = nn.Dropout(dropout) if dropout else nn.Identity()

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA+ : h = (I + (alpha/r) B A) W0 x, evaluated as y + s B(A y), y = W0 x.

    Multiplicative instead of additive. The identity term is the rank inflation
    (rank(I + BA) >= d - r), and computing W0 x first keeps the cost O(dkr)
    like LoRA. A maps from the OUTPUT dimension. B = 0 so step 0 is the
    pretrained function. Same implementation as Phase 2 (arXiv 2506.07621 eq. 8).
    """

    def __init__(self, base, r, alpha=16.0):
        super().__init__()
        self.base = base
        for q in self.base.parameters():
            q.requires_grad_(False)
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        nn.init.normal_(self.A, std=1 / math.sqrt(d))
        self.scale = alpha / r

    def forward(self, x):
        y = self.base(x)
        return y + F.linear(F.linear(y, self.A), self.B) * self.scale


# ----------------------------------------------------------------- QLoRA -----
NF4 = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen weight held as 4-bit NormalFloat codes, two per byte.

    Block-wise absmax (64 weights per block), and the absmax constants are
    themselves quantised to int8 in blocks of 256 around their mean -- QLoRA's
    "double quantisation" -- so the overhead is ~0.127 bits/weight rather than
    0.5. Dequantised on the fly into the compute dtype for each matmul. Storage
    is genuinely 4-bit; compute is bf16, as in the paper.
    """

    BLOCK, DQ_BLOCK = 64, 256

    def __init__(self, base: nn.Linear):
        super().__init__()
        w = base.weight.detach().float().cpu()
        self.out_features, self.in_features = w.shape
        flat = w.flatten()
        n = flat.numel()
        pad = (-n) % self.BLOCK
        if pad:
            flat = torch.cat([flat, flat.new_zeros(pad)])
        blocks = flat.view(-1, self.BLOCK)
        absmax = blocks.abs().amax(1).clamp(min=1e-12)
        normed = blocks / absmax[:, None]
        mid = (NF4[1:] + NF4[:-1]) / 2
        codes = torch.bucketize(normed, mid).to(torch.uint8).flatten()
        packed = (codes[0::2] << 4) | codes[1::2]
        # double quantisation of absmax
        mean = absmax.mean()
        c = absmax - mean
        dpad = (-c.numel()) % self.DQ_BLOCK
        cpad = torch.cat([c, c.new_zeros(dpad)]).view(-1, self.DQ_BLOCK)
        dscale = cpad.abs().amax(1).clamp(min=1e-12) / 127
        q8 = torch.round(cpad / dscale[:, None]).clamp(-127, 127).to(torch.int8)
        self.register_buffer("packed", packed)
        self.register_buffer("q8", q8)
        self.register_buffer("dscale", dscale)
        self.register_buffer("mean", mean.reshape(1))
        self.register_buffer("levels", NF4.clone())
        self.n, self.nblk = n, blocks.size(0)
        deq = self.dequant(torch.float32)
        self.rel_err = float((deq - w).norm() / w.norm())

    def dequant(self, dtype):
        hi = (self.packed >> 4).long()
        lo = (self.packed & 0x0F).long()
        codes = torch.stack([hi, lo], 1).flatten()
        absmax = (self.q8.float() * self.dscale[:, None]).flatten()[:self.nblk] + self.mean
        w = self.levels[codes].view(-1, self.BLOCK) * absmax[:, None]
        return w.flatten()[:self.n].view(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype))


# --------------------------------------------------------------- AdaLoRA -----
class SVDLinear(nn.Module):
    """y = W x + (alpha/r) P diag(E) Q x. E starts at 0; pruning zeroes entries of E."""

    def __init__(self, base, r, alpha=16.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.E = nn.Parameter(torch.zeros(r))
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.scale = alpha / r
        self.register_buffer("keep", torch.ones(r))

    def forward(self, x):
        e = self.E * self.keep
        return self.base(x) + F.linear(F.linear(x, self.Q) * e, self.P) * self.scale


class AdaLoRAController:
    """Sensitivity-smoothed importance and a cubic rank-budget schedule.

    Importance of triplet i in a layer (paper eq. 6-9):
        I(w)  = |w * dL/dw|,   Ibar <- b1 Ibar + (1-b1) I,
        U     = |I - Ibar|,    Ubar <- b2 Ubar + (1-b2) U,   s(w) = Ibar * Ubar
        S_i   = s(E_i) + mean_j s(P_ji) + mean_j s(Q_ij)
    The total budget falls cubically from r_init * n_layers to r_target * n_layers
    between t_init and T - t_final; every `every` steps the lowest-scoring
    triplets across ALL layers are masked, so layers that matter keep more rank.
    Orthogonality of P and Q is encouraged with the paper's regulariser.
    """

    def __init__(self, mods, r_init, r_target, total, t_init_frac=0.1,
                 t_final_frac=0.3, every=10, b1=0.85, b2=0.85, orth=0.1):
        self.mods = mods
        self.r_init, self.r_target = r_init, r_target
        self.total = total
        self.t_init = int(t_init_frac * total)
        self.t_final = int(t_final_frac * total)
        self.every, self.b1, self.b2, self.orth = every, b1, b2, orth
        self.state = {}
        self.budget_log = []

    def orth_loss(self):
        reg = 0.0
        for m in self.mods:
            r = m.E.numel()
            eye = torch.eye(r, device=m.P.device)
            reg = reg + ((m.P.T @ m.P - eye) ** 2).sum() + ((m.Q @ m.Q.T - eye) ** 2).sum()
        return self.orth * reg / max(1, len(self.mods))

    def budget(self, step):
        b0 = self.r_init * len(self.mods)
        bt = self.r_target * len(self.mods)
        if step <= self.t_init:
            return b0
        if step >= self.total - self.t_final:
            return bt
        frac = (step - self.t_init) / max(1, self.total - self.t_final - self.t_init)
        return int(bt + (b0 - bt) * (1 - frac) ** 3)

    @torch.no_grad()
    def step(self, step):
        """Call after loss.backward(), before zero_grad."""
        scores = []
        for k, m in enumerate(self.mods):
            for name in ("P", "E", "Q"):
                p = getattr(m, name)
                if p.grad is None:
                    continue
                ipt = (p * p.grad).abs()
                key = (k, name)
                if key not in self.state:
                    self.state[key] = [torch.zeros_like(ipt), torch.zeros_like(ipt)]
                ib, ub = self.state[key]
                ib.mul_(self.b1).add_(ipt, alpha=1 - self.b1)
                ub.mul_(self.b2).add_((ipt - ib).abs(), alpha=1 - self.b2)
            if (k, "E") in self.state:
                s = lambda n: self.state[(k, n)][0] * self.state[(k, n)][1]
                scores.append(s("E") + s("P").mean(0) + s("Q").mean(1))
            else:
                scores.append(None)
        if step <= self.t_init or step % self.every or any(s is None for s in scores):
            return
        b = self.budget(step)
        allsc = torch.cat(scores)
        if b >= allsc.numel():
            return
        thr = torch.kthvalue(allsc.float(), allsc.numel() - b).values
        for m, s in zip(self.mods, scores):
            m.keep.copy_((s > thr).float())
        self.budget_log.append((step, b))

    def ranks(self):
        return [int(m.keep.sum()) for m in self.mods]


# ------------------------------------------------------------------- IA3 -----
class ScaleOut(nn.Module):
    """IA3 on keys/values: y = l * (W x)."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        return self.base(x) * self.l


class ScaleIn(nn.Module):
    """IA3 on the FFN: rescale the intermediate activation before W_down."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l)


# ------------------------------------------------------------ prefix/prompt --
class PrefixEncoder(nn.Module):
    """P learned states for every layer, through Li & Liang's MLP reparametrisation.

    The states are written into the first P positions of each layer's INPUT, so
    every real token attends to keys/values computed from them. With
    [W_k; W_v] full row rank, any key/value pair is reachable from some state,
    so this is prefix-tuning, not shallow prompt-tuning; it differs from the
    original only in that RoPE and qk-norm also apply to the prefix keys.
    """

    def __init__(self, n_layer, n_prefix, d, hidden=512):
        super().__init__()
        self.n_layer, self.n_prefix, self.d = n_layer, n_prefix, d
        self.emb = nn.Parameter(torch.randn(n_prefix, d) * 0.02)
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.Tanh(),
                                 nn.Linear(hidden, n_layer * d))
        nn.init.normal_(self.mlp[2].weight, std=0.02)
        nn.init.zeros_(self.mlp[2].bias)

    def forward(self):
        return self.mlp(self.emb).view(self.n_prefix, self.n_layer, self.d).transpose(0, 1)


# ---------------------------------------------------------------- Phase 4 ----
class BiasLinear(nn.Module):
    """BitFit on a bias-free Linear: y = W x + b, W frozen, b zero-initialised."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.bias = nn.Parameter(torch.zeros(base.out_features))

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features

    def forward(self, x):
        return self.base(x) + self.bias.to(x.dtype)


class HoulsbyAdapter(nn.Module):
    """y = W x;  y + up(gelu(down(LN(y)))).  up = 0 so step 0 is the pretrained net.

    Placed on the attention output projection and on the FFN down-projection,
    i.e. after each sub-layer and before its residual add (Houlsby 2019, fig. 2).
    """

    def __init__(self, base, m=64):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        d = base.out_features
        self.ln = nn.LayerNorm(d)
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    @property
    def in_features(self):
        return self.base.in_features

    @property
    def out_features(self):
        return self.base.out_features

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(self.ln(y))))


class UnfreezeController:
    """Gradual unfreezing: block i (counted from the top) opens at step i*every.

    Parameters of a closed block have requires_grad False, get no gradient and
    are skipped by AdamW, so a closed block costs no optimiser work; its
    backward is still computed below the lowest open block only when needed.
    """

    def __init__(self, blocks, every, extra=()):
        self.blocks = list(blocks)[::-1]          # top first
        self.every = max(1, every)
        self.n_open = 0
        self.extra = list(extra)                  # final norm etc., opened with block 0
        self.total = 1
        self.t_init = self.t_final = 0
        self._open(1)

    def _open(self, k):
        k = min(k, len(self.blocks))
        while self.n_open < k:
            for p in self.blocks[self.n_open].parameters():
                p.requires_grad_(True)
            if self.n_open == 0:
                for p in self.extra:
                    p.requires_grad_(True)
            self.n_open += 1

    def orth_loss(self):
        return 0.0

    def step(self, step):
        self._open(1 + (step + 1) // self.every)

    def ranks(self):
        return [self.n_open]


# ------------------------------------------------------------------ apply -----
def blocks_of(trunk):
    return list(getattr(trunk, "blocks", None) or getattr(trunk, "enc"))


def slots(block):
    """(parent, attr, role) for every linear layer in one block."""
    out = []
    att = block.attn
    for name, role in (("wq", "q"), ("wk", "k"), ("wv", "v"), ("wo", "o"),
                       ("kv_down", "kvd"), ("k_up", "k"), ("v_up", "v")):
        if isinstance(getattr(att, name, None), nn.Module):
            out.append((att, name, role))
    if hasattr(block, "w_gate"):
        out += [(block, "w_gate", "gate"), (block, "w_up", "up"), (block, "w_down", "down")]
    elif hasattr(block, "ffn"):
        out += [(block.ffn, "g", "gate"), (block.ffn, "u", "up"), (block.ffn, "d", "down")]
    return out


def _get(parent, attr):
    return parent[attr] if isinstance(parent, nn.ModuleDict) else getattr(parent, attr)


def _set(parent, attr, mod):
    if isinstance(parent, nn.ModuleDict):
        parent[attr] = mod
    else:
        setattr(parent, attr, mod)


TARGETS = {
    "qv": {"q", "v"},
    "attn": {"q", "k", "v", "o", "kvd"},
    "all": {"q", "k", "v", "o", "kvd", "gate", "up", "down"},
}


def apply_method(model, a):
    """Freeze the trunk, insert adapters, and return (n_trainable, controller, info)."""
    trunk = model.trunk
    for p in trunk.parameters():
        p.requires_grad_(False)
    m = a.method
    ctrl, info = None, {}
    blocks = blocks_of(trunk)
    tg = TARGETS[a.targets]

    if m == "full":
        for p in trunk.parameters():
            p.requires_grad_(True)
    elif m in ("head", "prompt", "prefix"):
        pass
    elif m == "norm":
        for n, p in trunk.named_parameters():
            if ".n1." in n or ".n2." in n or ".n3." in n or n.startswith("norm") \
                    or n.startswith("enc_norm") or "kv_norm" in n:
                p.requires_grad_(True)
    elif m in ("lora", "rslora"):
        for b in blocks:
            for par, at, role in slots(b):
                if role in tg:
                    _set(par, at, LoRALinear(_get(par, at), a.rank, a.alpha,
                                             rs=(m == "rslora"), dropout=a.lora_dropout))
    elif m == "qlora":
        errs = []
        for b in blocks:
            for par, at, role in slots(b):
                q = NF4Linear(_get(par, at))
                errs.append(q.rel_err)
                _set(par, at, LoRALinear(q, a.rank, a.alpha, dropout=a.lora_dropout)
                     if role in TARGETS["all"] else q)
        info["nf4_rel_err_mean"] = sum(errs) / len(errs)
    elif m == "adalora":
        mods = []
        for b in blocks:
            for par, at, role in slots(b):
                if role in tg:
                    s = SVDLinear(_get(par, at), a.rank, a.alpha)
                    _set(par, at, s)
                    mods.append(s)
        ctrl = AdaLoRAController(mods, a.rank, a.ada_target, total=1)  # total set by trainer
    elif m == "lorma":
        for b in blocks:
            for par, at, role in slots(b):
                if role in TARGETS["qv"]:
                    _set(par, at, LoRMALinear(_get(par, at), a.rank, a.alpha))
    elif m == "bitfit":
        for b in blocks:
            for par, at, role in slots(b):
                _set(par, at, BiasLinear(_get(par, at)))
    elif m == "adapter":
        for b in blocks:
            for par, at, role in slots(b):
                if role in ("o", "down"):
                    _set(par, at, HoulsbyAdapter(_get(par, at), a.adapter_dim))
    elif m == "partial":
        k = a.top_k
        for b in blocks[-k:]:
            for p in b.parameters():
                p.requires_grad_(True)
        for p in final_norm_params(trunk):
            p.requires_grad_(True)
        info["top_k"] = k
    elif m == "unfreeze":
        ctrl = UnfreezeController(blocks, every=1, extra=final_norm_params(trunk))
        info["unfreeze"] = "top block first"
    elif m == "ia3":
        for b in blocks:
            for par, at, role in slots(b):
                if role in ("k", "v"):
                    _set(par, at, ScaleOut(_get(par, at)))
                elif role == "down":
                    _set(par, at, ScaleIn(_get(par, at)))
    else:
        raise ValueError(m)

    n_trunk = sum(p.numel() for p in trunk.parameters() if p.requires_grad)
    n_virt = sum(p.numel() for n, p in model.named_parameters()
                 if p.requires_grad and n.startswith(("prompt", "prefix")))
    n_head = sum(p.numel() for n, p in model.named_parameters()
                 if p.requires_grad and not n.startswith(("trunk.", "prompt", "prefix")))
    info.update(trainable_trunk=n_trunk + n_virt, trainable_head=n_head)
    return n_trunk + n_virt + n_head, ctrl, info


def final_norm_params(trunk):
    out = []
    for name in ("norm", "enc_norm"):
        mod = getattr(trunk, name, None)
        if isinstance(mod, nn.Module):
            out += list(mod.parameters())
    return out


def depth_of(model):
    """{id(param): depth index of its block (0 = bottom)}; non-block params -> None."""
    out = {}
    for i, b in enumerate(blocks_of(model.trunk)):
        for p in b.parameters():
            out[id(p)] = i
    return out


def trunk_param_ids(model):
    """ids of trainable parameters that belong to the trunk/adapters, for the LR split."""
    ids = {id(p) for p in model.trunk.parameters() if p.requires_grad}
    for name in ("prompt", "prefix"):
        mod = getattr(model, name, None)
        if mod is not None:
            ids |= {id(p) for p in (mod.parameters() if isinstance(mod, nn.Module) else [mod])}
    return ids

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/phase4_cjpe.py
#!/usr/bin/env python3
"""CS-781 Phase 4 -- court judgment prediction on a Phase-1 trunk.

    python phase4_cjpe.py --model M4_LexArcP  --method lorma   --tag m4_lorma
    python phase4_cjpe.py --model M2_LexArcED --method full    --tag m2_full
    python phase4_cjpe.py --model M1_SC       --method adapter --tag m1_adapter
    python phase4_cjpe.py --model M2_LexArcED --method full --keep-every 2 \
        --kd results/ens_T/teacher.pt --tag kd_m2_half
    python phase4_cjpe.py ... --bench          # 30 steps, print tokens/s and peak memory, exit

THE MODEL

    judgment (BPE ids, whole document tokenised once and cached)
      -> a VIEW picks K chunks of L tokens:
            tail      the last K*L tokens            (default; ILDC: the end carries the decision)
            head      the first K*L tokens
            headtail  1 chunk from the start + K-1 from the end
            spread    K chunks evenly spaced over the whole document
      -> Phase-1 trunk over each chunk (bidirectional attention unless --causal)
      -> masked mean over each chunk's tokens               -> one vector per chunk
      -> + embedding of the chunk's position decile in the document
      -> document pooling over chunks (--pool attn | mean | tf = 1-layer transformer + attn)
      -> LayerNorm -> dropout -> linear -> 2 logits

PROTOCOL

Train on a fixed, seeded subset of train.json (--train-docs; the same subset for
every run so runs are comparable), for --epochs passes (default 1: there are
tens of thousands of documents). Dev is evaluated --evals times over training;
the best-dev point's dev and test logits are saved. Selection metric is dev
Macro-F1 at the dev-tuned threshold. Everything the ensembler needs is in
results/<tag>/ (see phase4_data.py), so all ensembling/thresholding is CPU work.

KD (--kd teacher.pt): the student is trained on the KD pool -- train documents
the teachers NEVER saw (perm[train_docs : train_docs + kd_pool]), so the soft
labels are honest held-out predictions -- with gold CE + KL to the teacher, and
on the unlabelled test text with KL only (--kd-test).
"""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from phase4_data import (P1, P4, log as _log, token_cache, summarise, save_run,  # noqa: E402
                         write_submission, best_threshold, atomic_json)
from phase4_peft import (apply_method, blocks_of, trunk_param_ids, depth_of,     # noqa: E402
                         UnfreezeController, NF4Linear)
sys.path.insert(0, f"{P1}/shared/lib")

LOGF = None
PERM_SEED = 781


def log(*a):
    _log(*a)
    if LOGF:
        print(*a, file=LOGF, flush=True)


# ------------------------------------------------------------------- trunk --
def load_trunk(name, ckpt, keep_every=1, path=None):
    from cs781_model import Config, LexArcD
    ck = torch.load(path or f"{P1}/{name}/ckpt/{ckpt}", map_location="cpu", weights_only=False)
    cfg_d = ck["cfg"]
    sd = ck["model"] if "model" in ck else ck["state"]
    if "n_enc_layer" in cfg_d:
        from cs781_model_ed import EDConfig, LexArcED
        cfg = EDConfig(**cfg_d)
        t = LexArcED(cfg)
        t.load_state_dict(sd, strict=False)
        t.dec = nn.ModuleList()                 # only the encoder is used
        for attr in ("dec_norm", "head"):
            if hasattr(t, attr):
                delattr(t, attr)
        kind = "ed"
        if keep_every > 1:
            t.enc = nn.ModuleList(list(t.enc)[::keep_every])
    else:
        cfg = Config(**cfg_d)
        if getattr(cfg, "use_pointer", False):
            from cs781_model_ptr import LexArcP
            t = LexArcP(cfg)
        else:
            t = LexArcD(cfg)
        t.load_state_dict(sd, strict=False)
        for attr in ("head", "cq", "ck", "cv", "gate"):
            if hasattr(t, attr):
                delattr(t, attr)
        kind = "dec"
        if keep_every > 1:
            # Layer 0 must stay first: it produces the value-residual v0 that
            # every later layer's gate mixes in.
            t.blocks = nn.ModuleList(list(t.blocks)[::keep_every])
    return t, kind, cfg


# ------------------------------------------------------------------- model --
class CJPEModel(nn.Module):
    def __init__(self, trunk, kind, cfg, a):
        super().__init__()
        self.trunk, self.kind, self.a = trunk, kind, a
        d = cfg.d_model
        self.d = d
        self.R = getattr(cfg, "n_register", 0) if kind == "dec" else 0
        self.bi = not a.causal
        self.drop = nn.Dropout(a.dropout)
        self.cpos = nn.Embedding(10, d)
        nn.init.normal_(self.cpos.weight, std=0.02)
        if a.pool == "tf":
            layer = nn.TransformerEncoderLayer(d, 8, 2 * d, a.dropout, batch_first=True, norm_first=True)
            self.mix = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        else:
            self.mix = None
        self.att = nn.Sequential(nn.Linear(d, 256), nn.Tanh(), nn.Linear(256, 1))
        self.out_norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, 2)

    def token_hidden(self, idx, keep):
        trunk, B = self.trunk, idx.size(0)
        x = trunk.emb(idx)
        lead = 0
        if self.kind == "dec" and self.R:
            x = torch.cat([trunk.reg.expand(B, -1, -1).to(x.dtype), x], 1)
            lead = self.R
        T = x.size(1)
        from cs781_model import build_rope
        cos, sin = build_rope(T, trunk.cfg.head_dim, x.device, trunk.emb.weight.dtype,
                              trunk.cfg.rope_theta)
        kfull = torch.cat([keep.new_ones(B, lead), keep], 1) if lead else keep
        mask = kfull[:, None, None, :] if (self.kind == "ed" or self.bi) else None
        ckpt = self.training and self.a.grad_ckpt and torch.is_grad_enabled()

        def call(fn, *args):
            if ckpt:
                return torch.utils.checkpoint.checkpoint(fn, *args, use_reentrant=False)
            return fn(*args)

        if self.kind == "dec":
            v0 = None
            for i, blk in enumerate(trunk.blocks):
                if v0 is None:
                    x, v = call(lambda h, b=blk: b(h, cos, sin, None, None, mask)[:2], x)
                else:
                    x, v = call(lambda h, r, b=blk: b(h, cos, sin, r, None, mask)[:2], x, v0)
                if i == 0:
                    v0 = v
            x = trunk.norm(x)
        else:
            from cs781_model import apply_rope
            from cs781_model_ed import _apply_ffn

            def enc_layer(x, blk):
                at = blk.attn
                hN = blk.n1(x)
                q = at.wq(hN).view(B, T, at.nh, at.hd).transpose(1, 2)
                k = at.wk(hN).view(B, T, at.nkv, at.hd).transpose(1, 2)
                v = at.wv(hN).view(B, T, at.nkv, at.hd).transpose(1, 2)
                q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
                if at.cfg.qk_norm:
                    q = F.rms_norm(q, (at.hd,))
                    k = F.rms_norm(k, (at.hd,))
                y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, enable_gqa=True)
                x = x + at.wo(y.transpose(1, 2).reshape(B, T, at.nh * at.hd))
                return x + _apply_ffn(blk.ffn, blk.n2(x))

            for blk in trunk.enc:
                x = call(lambda h, b=blk: enc_layer(h, b), x)
            x = trunk.enc_norm(x)
        return x[:, lead:]

    def forward(self, bt):
        H = self.token_hidden(bt["idx"], bt["keep"])                 # (C, T, d)
        k = bt["keep"].unsqueeze(-1).to(H.dtype)
        vec = (H * k).sum(1).float() / k.sum(1).float().clamp(min=1)  # (C, d)
        B, K = bt["cvalid"].shape
        X = vec.new_zeros(B * K, self.d)
        X[bt["cslot"]] = vec
        X = X.view(B, K, self.d) + self.cpos(bt["cpos"]).float()
        valid = bt["cvalid"]
        if self.mix is not None:
            X = self.mix(X, src_key_padding_mask=~valid)
        if self.a.pool == "mean":
            w = valid.float() / valid.float().sum(1, keepdim=True).clamp(min=1)
        else:
            s = self.att(X).squeeze(-1).masked_fill(~valid, -1e4)
            w = torch.softmax(s, -1)
        self.last_w = w.detach()
        doc = (w.unsqueeze(-1) * X).sum(1)
        return self.head(self.drop(self.out_norm(doc)))           # (B, 2)


# -------------------------------------------------------------------- data --
def view_chunks(tok, a, train=False, rng=None):
    """[(start, end)] token spans for one document, and each span's position decile."""
    n = len(tok)
    L, K = a.chunk, a.chunks
    if n == 0:
        return [(0, 0)], [0]
    if a.view == "tail":
        spans = [(max(0, n - (j + 1) * L), n - j * L) for j in range(K) if n - j * L > 0]
        spans = spans[::-1]
    elif a.view == "head":
        spans = [(j * L, min(n, (j + 1) * L)) for j in range(K) if j * L < n]
    elif a.view == "headtail":
        spans = [(0, min(n, L))]
        rest = [(max(L, n - (j + 1) * L), n - j * L) for j in range(K - 1) if n - j * L > L]
        spans += rest[::-1]
    elif a.view == "spread":
        if n <= K * L:
            spans = [(j * L, min(n, (j + 1) * L)) for j in range(K) if j * L < n]
        else:
            starts = np.linspace(0, n - L, K).round().astype(int)
            spans = [(int(s), int(s) + L) for s in starts]
    else:
        raise ValueError(a.view)
    if train and a.jitter and rng is not None and a.view == "tail" and n > K * L:
        # shift the whole tail window back by up to `jitter` tokens: the end of a
        # judgment is formulaic, so this stops the model keying on exact offsets
        sh = rng.randint(0, min(a.jitter, n - K * L))
        spans = [(s - sh, e - sh) for s, e in spans]
    pos = [min(9, int(10 * s / max(1, n))) for s, _ in spans]
    return spans, pos


def collate(docs, toks, a, device, train=False, rng=None):
    """docs: list of indices into toks. -> batch dict on device."""
    seqs, cslot, cpos_l = [], [], []
    K = a.chunks
    for b, i in enumerate(docs):
        spans, pos = view_chunks(toks[i], a, train, rng)
        for j, ((s, e), p) in enumerate(zip(spans, pos)):
            seqs.append(toks[i][s:e])
            cslot.append(b * K + j)
            cpos_l.append((b, j, p))
    T = max(1, max(len(s) for s in seqs))
    C = len(seqs)
    idx = torch.zeros(C, T, dtype=torch.long)
    keep = torch.zeros(C, T, dtype=torch.bool)
    for j, s in enumerate(seqs):
        if len(s):
            idx[j, :len(s)] = torch.from_numpy(s.astype(np.int64))
            keep[j, :len(s)] = True
        else:
            keep[j, 0] = True
    B = len(docs)
    cvalid = torch.zeros(B, K, dtype=torch.bool)
    cpos = torch.zeros(B, K, dtype=torch.long)
    for b, j, p in cpos_l:
        cvalid[b, j] = True
        cpos[b, j] = p
    bt = dict(idx=idx, keep=keep, cvalid=cvalid, cpos=cpos, cslot=torch.tensor(cslot))
    bt = {k: v.to(device, non_blocking=True) for k, v in bt.items()}
    bt["ntok"] = int(keep.sum())
    return bt


@torch.no_grad()
def predict(model, toks, a, device, bs=None):
    """z = logit1 - logit0 for every document in toks (list of arrays)."""
    model.eval()
    bs = bs or max(1, a.batch * 2)
    order = sorted(range(len(toks)), key=lambda i: min(len(toks[i]), a.chunk * a.chunks))
    z = torch.zeros(len(toks))
    ntok, t0 = 0, time.time()
    for k in range(0, len(order), bs):
        part = order[k:k + bs]
        bt = collate(part, toks, a, device)
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            lo = model(bt).float().cpu()
        z[part] = lo[:, 1] - lo[:, 0]
        ntok += bt["ntok"]
    model.train()
    return z, ntok / max(1e-6, time.time() - t0)


# ------------------------------------------------------------------- train --
def build_model(a, device):
    trunk, kind, cfg = load_trunk(a.model, a.ckpt, a.keep_every, a.trunk_path)
    model = CJPEModel(trunk, kind, cfg, a)
    n_train, ctrl, info = apply_method(model, a)
    model = model.to(device)          # AFTER wrapping: adapters are new modules
    n_trunk = sum(p.numel() for p in model.trunk.parameters())
    n_trunk += sum(m.n for m in model.trunk.modules() if isinstance(m, NF4Linear))
    return model, ctrl, info, n_train, n_trunk, len(blocks_of(trunk))


def param_groups(model, a):
    """Head LR for the new modules, trunk LR for the trunk, optional layer-wise decay."""
    tid = trunk_param_ids(model)
    if a.method == "unfreeze":            # closed blocks must already be in the optimiser
        tid = {id(p) for p in model.trunk.parameters()}
    head = [p for p in model.parameters() if p.requires_grad and id(p) not in tid]
    groups = [dict(params=head, lr=a.head_lr, base=a.head_lr, name="head")]
    trunk = [p for p in model.trunk.parameters() if id(p) in tid and
             (p.requires_grad or a.method == "unfreeze")]
    if not trunk:
        return groups
    if a.llrd < 1.0:
        dep = depth_of(model)
        nb = len(blocks_of(model.trunk))
        by = {}
        for p in trunk:
            dpt = dep.get(id(p))
            f = 1.0 if dpt is None else a.llrd ** (nb - 1 - dpt)
            if dpt is None and p is model.trunk.emb.weight:
                f = a.llrd ** nb
            by.setdefault(f, []).append(p)
        for f, ps in sorted(by.items()):
            groups.append(dict(params=ps, lr=a.lr * f, base=a.lr * f, name=f"trunk x{f:.3f}"))
    else:
        groups.append(dict(params=trunk, lr=a.lr, base=a.lr, name="trunk"))
    return groups


def run(a, device, tr_toks, tr_y, dev_toks, dev_y, test_toks, kd=None, tr_w=None):
    torch.manual_seed(a.seed)
    rng = random.Random(a.seed)
    model, ctrl, info, n_train, n_trunk, n_blocks = build_model(a, device)
    log(f"  {a.method}: {n_train/1e6:.3f}M trainable | trunk {n_trunk/1e6:.1f}M "
        f"({n_blocks} layers) | {info}")
    groups = param_groups(model, a)
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), weight_decay=a.wd)
    params = [p for g in groups for p in g["params"]]
    # class weights from the training labels (gold only)
    ys = [y for y in tr_y if y is not None]
    pos = sum(ys) / max(1, len(ys))
    cw = torch.tensor([0.5 / max(1e-3, 1 - pos), 0.5 / max(1e-3, pos)]) if a.class_weight else torch.ones(2)
    cw = cw.to(device)
    log(f"  train docs {len(tr_toks)} (gold {len(ys)}, pos rate {pos:.3f}) class weights "
        f"{[round(x, 3) for x in cw.tolist()]}")

    n_docs = len(tr_toks)
    tr_w = tr_w or [1.0] * n_docs
    steps_per_ep = math.ceil(n_docs / (a.batch * a.accum))
    total = steps_per_ep * a.epochs if not a.max_steps else a.max_steps
    if isinstance(ctrl, UnfreezeController):
        ctrl.every = max(1, int(a.unfreeze_frac * total / n_blocks))
        log(f"  gradual unfreezing: one block every {ctrl.every} updates")
    elif ctrl is not None:
        ctrl.total = total
        ctrl.t_init, ctrl.t_final = int(0.1 * total), int(0.3 * total)
    warm = max(1, int(a.warmup * total))
    eval_at = sorted({max(1, round(total * (k + 1) / a.evals)) for k in range(a.evals)})
    if a.bench:
        eval_at = []                      # a benchmark measures training speed only
    log(f"  {total} updates ({a.batch}x{a.accum} docs, {a.chunks}x{a.chunk} tokens/doc), "
        f"dev evals at {eval_at}")

    best = dict(f1=-1.0)
    trace = []
    t0 = time.time()
    tok_seen, t_last, tok_last = 0, time.time(), 0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    step = 0
    done = False
    for ep in range(a.epochs if not a.max_steps else 10 ** 6):
        order = list(range(n_docs))
        rng.shuffle(order)
        micro = [order[k:k + a.batch] for k in range(0, len(order), a.batch)]
        tl, tn = 0.0, 0
        for mi, part in enumerate(micro):
            bt = collate(part, tr_toks, a, device, train=True, rng=rng)
            with torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                lo = model(bt)
            lo = lo.float()
            y = torch.tensor([tr_y[i] if tr_y[i] is not None else -1 for i in part], device=device)
            lab = y >= 0
            loss = lo.new_zeros(())
            if lab.any():
                sw = torch.tensor([tr_w[i] for i in part], device=device)[lab]
                ce_i = F.cross_entropy(lo[lab], y[lab], weight=cw, label_smoothing=a.smooth,
                                       reduction="none")
                ce = (ce_i * sw).sum() / (cw[y[lab]] * sw).sum().clamp(min=1e-6)
                loss = ce
            if kd is not None:
                tz = torch.tensor([kd[i] for i in part], device=device, dtype=torch.float32)
                T = a.kd_temp
                pt = torch.sigmoid(tz / T)
                pt = torch.stack([1 - pt, pt], -1)
                ls = F.log_softmax(lo / T, -1)
                kl = (pt * (pt.clamp(min=1e-9).log() - ls)).sum(-1) * T * T
                kdl = kl.mean()
                if lab.any():
                    # labelled docs: (1-alpha) CE + alpha KL; unlabelled: KL only
                    m = lab.float()
                    kdl_l = (kl * m).sum() / m.sum()
                    kdl_u = (kl * (1 - m)).sum() / (1 - m).sum().clamp(min=1)
                    loss = (1 - a.kd_alpha) * loss + a.kd_alpha * kdl_l + \
                        a.kd_alpha * kdl_u * ((1 - m).sum() / len(part))
                else:
                    loss = kdl
            if ctrl is not None and hasattr(ctrl, "orth_loss"):
                ol = ctrl.orth_loss()
                if torch.is_tensor(ol):
                    loss = loss + ol
            (loss / a.accum).backward()
            tok_seen += bt["ntok"]
            tl += float(loss.detach())
            tn += 1
            if (mi + 1) % a.accum and mi + 1 != len(micro):
                continue
            lr_f = (step / warm if step < warm else
                    0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm))))
            for g in opt.param_groups:
                g["lr"] = g["base"] * max(lr_f, a.min_lr_frac)
            torch.nn.utils.clip_grad_norm_([p for p in params if p.grad is not None], 1.0)
            opt.step()
            if ctrl is not None:
                ctrl.step(step)
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % a.log_every == 0 or step == 10:
                now = time.time()
                tps = (tok_seen - tok_last) / max(1e-6, now - t_last)
                t_last, tok_last = now, tok_seen
                mem = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0
                eta = (total - step) * (now - t0) / step / 60
                log(f"    step {step:5d}/{total} loss {tl/max(1,tn):.4f} | {tps/1e3:.1f}k tok/s "
                    f"| peak {mem:.1f} GB | {(now-t0)/60:.1f} min, ETA {eta:.0f} min")
                with open(f"{P4}/logs/live_{a.tag}.txt", "a", encoding="utf-8") as lf:
                    lf.write(f"{time.strftime('%H:%M')} step {step}/{total} {tps/1e3:.1f}k tok/s "
                             f"loss {tl/max(1,tn):.4f} ETA {eta:.0f}m\n")
                tl, tn = 0.0, 0
            if a.bench and step >= a.max_steps:
                done = True
                break
            if step in eval_at:
                zd, inf_tps = predict(model, dev_toks, a, device)
                s = summarise(dev_y, zd.tolist())
                trace.append(dict(step=step, **s, t=time.time() - t0))
                log(f"    [eval step {step}] dev macro-F1 @0 {s['f1_at0']:.4f} | tuned {s['f1_tuned']:.4f} "
                    f"(thr {s['thr']:+.2f}) | AUC {s['auc']:.4f} | P {s['precision']:.3f} R {s['recall']:.3f}"
                    f" | {inf_tps/1e3:.1f}k tok/s inference")
                with open(f"{P4}/logs/live_{a.tag}.txt", "a", encoding="utf-8") as lf:
                    lf.write(f"{time.strftime('%H:%M')} EVAL step {step} dev F1 {s['f1_tuned']:.4f} "
                             f"AUC {s['auc']:.4f}\n")
                if s["f1_tuned"] > best["f1"]:
                    t1 = time.time()
                    zt, _ = predict(model, test_toks, a, device)
                    best = dict(f1=s["f1_tuned"], step=step, dev=zd, test=zt, summ=s,
                                inf_s=time.time() - t1, inf_tps=inf_tps)
                    keys = {n for n, p in model.named_parameters() if p.requires_grad}
                    keys |= {n for n, _ in model.named_buffers() if n.endswith(".keep")}
                    best["state"] = {k: v.detach().to("cpu", copy=True) for k, v in
                                     model.state_dict().items() if k in keys}
            if step >= total:
                done = True
                break
        if done:
            break
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    train_s = time.time() - t0
    tps_all = tok_seen / max(1e-6, train_s)
    return model, best, trace, dict(n_trainable=n_train, n_trunk=n_trunk, n_blocks=n_blocks,
                                    info=info, peak_gb=peak, train_s=train_s,
                                    train_tok_per_s=tps_all, tokens_seen=tok_seen,
                                    ranks=ctrl.ranks() if ctrl is not None else None)


def predict_only(a, device, dv, te):
    """Re-score a finished run from its saved weights: dev for the threshold, test for the zip.

    For when the organisers replace test.json. Run it with the SAME arguments as
    the original run (the queue's command line) plus --predict-only.
    """
    cp = f"{P4}/ckpt/{a.tag}.pt"
    assert os.path.exists(cp), f"{cp} missing: this run was trained without --save-ckpt"
    ck = torch.load(cp, map_location="cpu", weights_only=False)
    model, _, _, _, _, _ = build_model(a, device)
    missing = model.load_state_dict({k: v.float() if v.is_floating_point() else v
                                     for k, v in ck["state"].items()}, strict=False)
    log(f"  {a.tag}: loaded {len(ck['state'])} tensors from {cp} "
        f"({len(missing.unexpected_keys)} unexpected)")
    zd, _ = predict(model, dv["tok"], a, device)
    zt, _ = predict(model, te["tok"], a, device)
    s = summarise(dv["label"], zd.tolist())
    rd = f"{P4}/results/{a.tag}"
    meta = json.load(open(f"{rd}/meta.json")) if os.path.exists(f"{rd}/meta.json") else dict(tag=a.tag)
    meta.update(dev_f1=s["f1_tuned"], thr=s["thr"], dev_auc=s["auc"], dev_p=s["precision"], dev_r=s["recall"],
                repredicted=time.strftime("%Y-%m-%d %H:%M"), n_test=len(te["ids"]))
    save_run(a.tag, dv["ids"], zd, te["ids"], zt, meta)
    log(f"  {a.tag}: re-predicted {len(te['ids'])} test docs | dev macro-F1 {s['f1_tuned']:.4f}")
    if a.submit:
        pred = {d: int(v > s["thr"]) for d, v in zip(te["ids"], zt.tolist())}
        write_submission(a.tag, te["ids"], pred, dev=s["f1_tuned"],
                         note=f"{a.model} {a.method} re-predicted from ckpt", kind="model")


# -------------------------------------------------------------------- main --
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="M1_SC M2_LexArcED M3_LexArcL M4_LexArcP")
    ap.add_argument("--ckpt", default="best.pt")
    ap.add_argument("--trunk-path", default=None)
    ap.add_argument("--method", default="full",
                    choices=["head", "full", "norm", "bitfit", "partial", "unfreeze", "adapter",
                             "lora", "rslora", "qlora", "adalora", "lorma", "ia3"])
    ap.add_argument("--rank", type=int, default=16)
    ap.add_argument("--alpha", type=float, default=32.0)
    ap.add_argument("--ada-target", type=int, default=6)
    ap.add_argument("--targets", default="attn", choices=["qv", "attn", "all"])
    ap.add_argument("--lora-dropout", type=float, default=0.05)
    ap.add_argument("--adapter-dim", type=int, default=64)
    ap.add_argument("--top-k", type=int, default=4, help="partial: trainable top blocks")
    ap.add_argument("--unfreeze-frac", type=float, default=0.5,
                    help="unfreeze: all blocks open after this fraction of training")
    ap.add_argument("--llrd", type=float, default=1.0, help="layer-wise LR decay per block (1 = off)")
    ap.add_argument("--keep-every", type=int, default=1, help="2 = half-depth student")
    ap.add_argument("--causal", action="store_true")
    ap.add_argument("--n-virtual", type=int, default=0)
    # document view
    ap.add_argument("--view", default="tail", choices=["tail", "head", "headtail", "spread"])
    ap.add_argument("--chunk", type=int, default=512)
    ap.add_argument("--chunks", type=int, default=2)
    ap.add_argument("--jitter", type=int, default=0)
    ap.add_argument("--pool", default="attn", choices=["attn", "mean", "tf"])
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--grad-ckpt", action="store_true")
    # optimisation
    ap.add_argument("--train-docs", type=int, default=12000)
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--batch", type=int, default=8, help="documents per micro-batch")
    ap.add_argument("--accum", type=int, default=2)
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--head-lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01)
    ap.add_argument("--warmup", type=float, default=0.06)
    ap.add_argument("--min-lr-frac", type=float, default=0.0)
    ap.add_argument("--smooth", type=float, default=0.0)
    ap.add_argument("--no-class-weight", dest="class_weight", action="store_false")
    ap.add_argument("--evals", type=int, default=3)
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--max-steps", type=int, default=0)
    ap.add_argument("--bench", action="store_true", help="run --max-steps (default 30) updates, report speed, exit")
    ap.add_argument("--use-rest", action="store_true",
                    help="also train on every train doc after the KD pool (all gold except the KD pool)")
    ap.add_argument("--extra", default=None, help="silver jsonl from phase4_external.py")
    ap.add_argument("--extra-docs", type=int, default=0, help="0 = all")
    ap.add_argument("--extra-weight", type=float, default=0.5)
    # KD
    ap.add_argument("--kd", default=None, help="teacher file {'ids': [...], 'z': tensor} over kd pool + test")
    ap.add_argument("--kd-pool", type=int, default=8000, help="train docs after --train-docs used for KD")
    ap.add_argument("--kd-alpha", type=float, default=0.7)
    ap.add_argument("--kd-temp", type=float, default=2.0)
    ap.add_argument("--kd-test", action="store_true", help="distil on the test text too")
    ap.add_argument("--dump-kd", action="store_true",
                    help="after training, predict the KD pool with the best state (teacher candidate)")
    # output
    ap.add_argument("--tag", required=True)
    ap.add_argument("--submit", action="store_true")
    ap.add_argument("--save-ckpt", action="store_true", help="keep the best trainable weights (bf16) in ckpt/<tag>.pt")
    ap.add_argument("--predict-only", action="store_true",
                    help="no training: load ckpt/<tag>.pt, re-predict dev + the CURRENT test.json, rewrite the zip")
    ap.add_argument("--force", action="store_true")
    a = ap.parse_args()
    if a.lr is None:
        a.lr = dict(full=3e-5, head=0.0, norm=1e-3, bitfit=1e-3, partial=5e-5, unfreeze=5e-5,
                    adapter=5e-4, lora=3e-4, rslora=2e-4, qlora=3e-4, adalora=5e-4, lorma=3e-4,
                    ia3=3e-3)[a.method]
    if a.bench and not a.max_steps:
        a.max_steps = 30
    a.n_virtual = 0

    global LOGF
    rd = f"{P4}/results/{a.tag}"
    if os.path.exists(f"{rd}/meta.json") and not a.force and not a.bench and not a.predict_only:
        log(f"{a.tag}: {rd}/meta.json exists, skipping (use --force)")
        return
    os.makedirs(f"{P4}/logs", exist_ok=True)
    LOGF = open(f"{P4}/logs/{a.tag}.log", "a", encoding="utf-8")
    if torch.cuda.is_available():
        device = torch.device("cuda")
        p = torch.cuda.get_device_properties(0)
        assert p.major * 10 + p.minor >= 80, f"{p.name}: bf16 is emulated below SM80 -- use an L4/A100"
        torch.set_float32_matmul_precision("high")
        gpu = p.name
    else:
        device, gpu = torch.device("cpu"), "cpu"
    log(f"\n===== {a.tag} | {gpu} | {a.model}/{a.ckpt} method={a.method} view={a.view} "
        f"{a.chunks}x{a.chunk} pool={a.pool} keep_every={a.keep_every} =====")
    log("  args: " + json.dumps(vars(a), sort_keys=True))

    from phase4_data import drop_test_copies
    tr = drop_test_copies(token_cache("train"), "train")
    dv = drop_test_copies(token_cache("dev"), "dev")
    te = token_cache("test")
    if a.predict_only:
        predict_only(a, device, dv, te)
        return
    perm = list(range(len(tr["ids"])))
    random.Random(PERM_SEED).shuffle(perm)
    sub = perm[:a.train_docs]
    kd = None
    if a.kd:
        teach = torch.load(a.kd, weights_only=False)
        tz = {d: float(v) for d, v in zip(teach["ids"], teach["z"].tolist())}
        pool = perm[a.train_docs:a.train_docs + a.kd_pool]
        tr_toks = [tr["tok"][i] for i in pool]
        tr_y = [tr["label"][i] for i in pool]
        kd = [tz[tr["ids"][i]] for i in pool]
        if a.kd_test:
            tr_toks += list(te["tok"])
            tr_y += [None] * len(te["ids"])
            kd += [tz[d] for d in te["ids"]]
        miss = sum(1 for v in kd if v is None)
        assert not miss, f"teacher missing {miss} docs"
        log(f"  KD: {len(pool)} held-out train docs + {len(te['ids']) if a.kd_test else 0} test docs, "
            f"alpha {a.kd_alpha} T {a.kd_temp}")
    else:
        if a.use_rest:
            sub = sub + perm[a.train_docs + a.kd_pool:]
        tr_toks = [tr["tok"][i] for i in sub]
        tr_y = [tr["label"][i] for i in sub]
    tr_w = [1.0] * len(tr_toks)
    if a.extra:
        assert not a.kd, "--extra with --kd is not supported (silver docs have no teacher scores)"
        from phase4_data import ext_cache
        ec = ext_cache(a.extra)
        k = a.extra_docs or len(ec["ids"])
        tr_toks += list(ec["tok"][:k])
        tr_y += list(ec["label"][:k])
        tr_w += [a.extra_weight] * min(k, len(ec["ids"]))
        log(f"  extra: {min(k, len(ec['ids']))} silver docs from {a.extra} at weight {a.extra_weight}")
    log(f"  training pool: {len(tr_toks)} docs")

    model, best, trace, cost = run(a, device, tr_toks, tr_y, dv["tok"], dv["label"], te["tok"], kd, tr_w)
    if a.bench:
        os.makedirs(f"{P4}/bench", exist_ok=True)
        b = dict(tag=a.tag, model=a.model, method=a.method, chunks=a.chunks, chunk=a.chunk,
                 batch=a.batch, grad_ckpt=a.grad_ckpt, keep_every=a.keep_every,
                 tok_per_s=cost["train_tok_per_s"], peak_gb=cost["peak_gb"], gpu=gpu,
                 n_trainable=cost["n_trainable"])
        atomic_json(b, f"{P4}/bench/{a.tag}.json", indent=1)
        log(f"  BENCH {a.tag}: {cost['train_tok_per_s']/1e3:.1f}k tok/s train, peak {cost['peak_gb']:.1f} GB")
        return

    s = best["summ"]
    tok_per_doc = a.chunks * a.chunk
    est = 12000 * tok_per_doc / max(1, cost["train_tok_per_s"]) / 60
    meta = dict(tag=a.tag, args=vars(a), dev_f1=s["f1_tuned"], dev_f1_at0=s["f1_at0"], thr=s["thr"],
                dev_auc=s["auc"], dev_p=s["precision"], dev_r=s["recall"], best_step=best["step"],
                trace=trace, kind="model", technique="kd" if a.kd else a.method, arch=a.model,
                **{k: v for k, v in cost.items()},
                test_inference_s=best["inf_s"], inference_tok_per_s=best["inf_tps"],
                minutes_per_12k_docs=est)
    extra = {}
    if a.dump_kd:
        model.load_state_dict({k: v.to(device) for k, v in best["state"].items()}, strict=False)
        pool = perm[a.train_docs:a.train_docs + a.kd_pool]
        zk, _ = predict(model, [tr["tok"][i] for i in pool], a, device)
        extra["kdpool"] = dict(ids=[tr["ids"][i] for i in pool], z=zk)
        yk = [tr["label"][i] for i in pool]
        sk = summarise(yk, zk.tolist(), thr=s["thr"])
        meta["kdpool_f1"] = sk["f1_tuned"]
        log(f"  KD pool ({len(pool)} unseen train docs): macro-F1 {sk['f1_tuned']:.4f} at the dev threshold "
            f"| AUC {sk['auc']:.4f}")
    save_run(a.tag, dv["ids"], best["dev"], te["ids"], best["test"], meta, extra)
    if a.save_ckpt:
        os.makedirs(f"{P4}/ckpt", exist_ok=True)
        st = {k: (v.to(torch.bfloat16) if v.is_floating_point() else v) for k, v in best["state"].items()}
        torch.save(dict(state=st, args=vars(a)), f"{P4}/ckpt/{a.tag}.pt")
    log(f"\n===== {a.tag}: dev macro-F1 {s['f1_tuned']:.4f} (thr {s['thr']:+.2f}; @0 {s['f1_at0']:.4f}) "
        f"AUC {s['auc']:.4f} | best step {best['step']} | {cost['n_trainable']/1e6:.2f}M trainable | "
        f"peak {cost['peak_gb']:.1f} GB | train {cost['train_s']/60:.1f} min "
        f"({cost['train_tok_per_s']/1e3:.1f}k tok/s) =====")
    if a.submit:
        zt = best["test"].tolist()
        pred = {d: int(v > s["thr"]) for d, v in zip(te["ids"], zt)}
        write_submission(a.tag, te["ids"], pred, dev=s["f1_tuned"],
                         note=f"{a.model} {a.method} {a.view} {a.chunks}x{a.chunk}", kind="model")
    try:
        from phase4_ens import write_status
        write_status()
    except Exception as e:
        log(f"  (status not updated: {e})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/phase4_ens.py
#!/usr/bin/env python3
"""CS-781 Phase 4 -- offline ensembling, thresholds, KD teacher, status. CPU only.

    python phase4_ens.py status
    python phase4_ens.py ens --runs all --kind model --greedy 8 --tag ens_model
    python phase4_ens.py ens --runs m4_lorma m2_full lex_tail --tag ens_hyb_3
    python phase4_ens.py ens --runs m2_full m4_lorma --teacher        # writes results/<tag>/teacher.pt
    python phase4_ens.py zip --run m4_lorma --rate                    # one run, rate-matched threshold

Every run is put on a common scale before averaging: z' = (z - thr) / sd_dev,
so each member's own dev-tuned decision boundary sits at 0 and a confident
model does not drown the others (the Phase-3 stacker failed on exactly this
kind of calibration difference). The averaged score is re-thresholded on dev.

--rate replaces the dev-tuned threshold with one that gives the test set the
dev set's predicted-positive rate (Phase 2: calibrating output volume moved the
leaderboard more than the CV-optimal cut did).
"""

from __future__ import annotations

import argparse
import glob
import json
import os
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from phase4_data import (P4, log, summarise, best_threshold, rate_threshold, save_run,  # noqa: E402
                         write_submission, load_run, metrics, token_cache, atomic_json)

REQUIRED = ["full", "partial", "unfreeze", "bitfit", "lorma", "adalora", "adapter"]
ARCHS = ["M1_SC", "M2_LexArcED", "M3_LexArcL", "M4_LexArcP"]


def all_runs():
    return sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(f"{P4}/results/*/meta.json"))


def dev_labels():
    if os.path.exists(f"{P4}/cache/tok_dev.pt"):
        dv = token_cache("dev")
        return {d: y for d, y in zip(dv["ids"], dv["label"])}
    from phase4_data import load_split
    return {d: y for d, _, y in load_split("dev")}


def scaled(tag, ydev):
    d, t, meta = load_run(tag)
    zd = d["z"].float()
    y = [ydev[i] for i in d["ids"]]
    thr, _ = best_threshold(y, zd.tolist())
    sd = float(zd.std()) or 1.0
    return (dict(zip(d["ids"], ((zd - thr) / sd).tolist())),
            dict(zip(t["ids"], ((t["z"].float() - thr) / sd).tolist())), meta)


def combine(tags, ydev, weights=None):
    S = [scaled(t, ydev) for t in tags]
    w = weights or [1.0] * len(tags)
    ids_d = list(S[0][0])
    ids_t = list(S[0][1])
    zd = [sum(wi * s[0][i] for wi, s in zip(w, S)) / sum(w) for i in ids_d]
    zt = [sum(wi * s[1][i] for wi, s in zip(w, S)) / sum(w) for i in ids_t]
    return ids_d, zd, ids_t, zt, [s[2] for s in S]


def dev_f1(ids_d, zd, ydev):
    y = [ydev[i] for i in ids_d]
    return best_threshold(y, zd)[1]


def cmd_ens(a):
    ydev = dev_labels()
    runs = all_runs() if a.runs == ["all"] else a.runs
    runs = [r for r in runs if not r.startswith("ens_") and not any(x in r for x in a.exclude)]
    if a.kind == "model":
        runs = [r for r in runs if json.load(open(f"{P4}/results/{r}/meta.json")).get("kind") == "model"]
    if a.teacher:
        runs = [r for r in runs if os.path.exists(f"{P4}/results/{r}/kdpool.pt")]
    assert runs, "no runs to ensemble"
    if a.greedy:
        # forward selection with replacement on dev Macro-F1 at the tuned threshold
        single = sorted(((dev_f1(*combine([r], ydev)[:2], ydev), r) for r in runs), reverse=True)
        log("  singles: " + ", ".join(f"{r} {f:.4f}" for f, r in single))
        chosen = [single[0][1]]
        cur = single[0][0]
        for _ in range(a.greedy - 1):
            cand = []
            for r in runs:
                f = dev_f1(*combine(chosen + [r], ydev)[:2], ydev)
                cand.append((f, r))
            f, r = max(cand)
            if f <= cur + a.min_gain:
                break
            chosen.append(r)
            cur = f
            log(f"  + {r}: dev {f:.4f}")
        runs = chosen
    ids_d, zd, ids_t, zt, metas = combine(runs, ydev)
    y = [ydev[i] for i in ids_d]
    s = summarise(y, zd)
    thr = s["thr"]
    if a.rate:
        thr = rate_threshold(zt, s["pos_rate"])
    kinds = {m.get("kind", "model") for m in metas}
    kind = "model" if kinds == {"model"} else "hybrid"
    from collections import Counter
    log(f"  {a.tag}: {len(runs)} members {dict(Counter(runs))} | dev macro-F1 {s['f1_tuned']:.4f} "
        f"(thr {s['thr']:+.3f}) AUC {s['auc']:.4f} | {kind}")
    meta = dict(tag=a.tag, members=runs, dev_f1=s["f1_tuned"], dev_auc=s["auc"], thr=thr,
                dev_p=s["precision"], dev_r=s["recall"], kind=kind, technique="ensemble",
                arch="+".join(sorted({m.get("arch", "?") for m in metas})), rate=a.rate)
    extra = {}
    if a.teacher:
        kp = None
        for r in runs:
            k = torch.load(f"{P4}/results/{r}/kdpool.pt", weights_only=False)
            d, _, _ = load_run(r)
            yd = [ydev[i] for i in d["ids"]]
            t0, _ = best_threshold(yd, d["z"].tolist())
            sd = float(d["z"].float().std()) or 1.0
            v = (k["z"].float() - t0) / sd
            kp = v if kp is None else kp + v
            kids = k["ids"]
        kp = kp / len(runs)
        # the teacher's z must be a calibrated logit for sigmoid(z/T): rescale the
        # averaged score so that its dev logistic fit is the identity
        import math
        zz = torch.tensor(zd)
        yy = torch.tensor(y, dtype=torch.float32)
        A = torch.zeros(2, requires_grad=True)
        opt = torch.optim.LBFGS([A], max_iter=100)

        def closure():
            opt.zero_grad()
            l = torch.nn.functional.binary_cross_entropy_with_logits(A[0] * zz + A[1], yy)
            l.backward()
            return l
        opt.step(closure)
        sa, sb = float(A[0]), float(A[1])
        log(f"  teacher calibration on dev: logit = {sa:.3f} * z + {sb:+.3f}")
        tz = {i: sa * v + sb for i, v in zip(kids, kp.tolist())}
        tz.update({i: sa * v + sb for i, v in zip(ids_t, zt)})
        teach = dict(ids=list(tz), z=torch.tensor(list(tz.values())))
        os.makedirs(f"{P4}/results/{a.tag}", exist_ok=True)
        torch.save(teach, f"{P4}/teacher_{a.tag}.pt")
        log(f"  wrote {P4}/teacher_{a.tag}.pt ({len(tz)} docs: KD pool + test)")
    save_run(a.tag, ids_d, zd, ids_t, zt, meta, extra)
    if a.submit:
        pred = {i: int(v > thr) for i, v in zip(ids_t, zt)}
        write_submission(a.tag, ids_t, pred, dev=s["f1_tuned"], kind=kind,
                         note=f"ensemble of {len(runs)}: {' '.join(runs)}" + (" rate-matched" if a.rate else ""))
    write_status()


def cmd_zip(a):
    ydev = dev_labels()
    d, t, meta = load_run(a.run)
    y = [ydev[i] for i in d["ids"]]
    s = summarise(y, d["z"].tolist())
    thr = s["thr"]
    zt = t["z"].tolist()
    if a.rate:
        thr = rate_threshold(zt, s["pos_rate"])
    tag = a.tag or (a.run + ("_rate" if a.rate else ""))
    pred = {i: int(v > thr) for i, v in zip(t["ids"], zt)}
    write_submission(tag, t["ids"], pred, dev=s["f1_tuned"], kind=meta.get("kind", "model"),
                     note=f"{a.run} thr {thr:+.3f}" + (" rate-matched" if a.rate else ""))


def write_status():
    """STATUS.md on Drive: every run, its dev score, cost, and requirement coverage."""
    rows = []
    for r in all_runs():
        m = json.load(open(f"{P4}/results/{r}/meta.json"))
        rows.append(m)
    rows.sort(key=lambda m: -m.get("dev_f1", 0))
    L = ["| run | kind | technique | arch | dev F1 | AUC | P | R | trainable M | peak GB | train min | tok/s |",
         "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for m in rows:
        L.append(f"| {m['tag']} | {m.get('kind','?')} | {m.get('technique','?')} | {m.get('arch','?')} | "
                 f"{m.get('dev_f1',0):.4f} | {m.get('dev_auc',float('nan')):.4f} | {m.get('dev_p',0):.3f} | "
                 f"{m.get('dev_r',0):.3f} | {m.get('n_trainable',0)/1e6:.2f} | {m.get('peak_gb',0):.1f} | "
                 f"{m.get('train_s',0)/60:.1f} | {m.get('train_tok_per_s',0)/1e3:.1f}k |")
    model_runs = [m for m in rows if m.get("kind") == "model" and m.get("technique") != "ensemble"]
    tech = {m["technique"] for m in model_runs}
    tech |= {"kd" for m in model_runs if (m.get("args") or {}).get("kd")}
    arch = {m["arch"] for m in model_runs}
    L.append("")
    L.append(f"techniques covered: {sorted(tech)}; missing from the phase list: "
             f"{[t for t in REQUIRED + ['kd'] if t not in tech]}")
    L.append(f"architectures covered: {sorted(arch)}; missing: {[x for x in ARCHS if x not in arch]}")
    sub = f"{P4}/submissions/tracker.json"
    if os.path.exists(sub):
        tr = json.load(open(sub))
        L.append("")
        L.append("| zip | kind | dev F1 | test pos-rate | note |")
        L.append("|---|---|---|---|---|")
        for t in sorted(tr, key=lambda t: -(t.get("dev") or 0)):
            L.append(f"| {t['zip']} | {t['kind']} | {t.get('dev') or 0:.4f} | {t['pos_rate']:.3f} | {t['note']} |")
    with open(f"{P4}/STATUS.md", "w", encoding="utf-8") as f:
        f.write("\n".join(L) + "\n")
    return "\n".join(L)


def main():
    ap = argparse.ArgumentParser()
    sp = ap.add_subparsers(dest="cmd", required=True)
    e = sp.add_parser("ens")
    e.add_argument("--runs", nargs="+", default=["all"])
    e.add_argument("--exclude", nargs="*", default=[])
    e.add_argument("--kind", choices=["any", "model"], default="any")
    e.add_argument("--greedy", type=int, default=0)
    e.add_argument("--min-gain", type=float, default=0.0005)
    e.add_argument("--rate", action="store_true")
    e.add_argument("--teacher", action="store_true")
    e.add_argument("--no-submit", dest="submit", action="store_false")
    e.add_argument("--tag", required=True)
    z = sp.add_parser("zip")
    z.add_argument("--run", required=True)
    z.add_argument("--rate", action="store_true")
    z.add_argument("--tag", default=None)
    sp.add_parser("status")
    a = ap.parse_args()
    if a.cmd == "ens":
        if not a.tag.startswith("ens_"):
            a.tag = "ens_" + a.tag
        if a.teacher:
            a.submit = False             # a teacher is an input to KD, not a submission
        cmd_ens(a)
    elif a.cmd == "zip":
        cmd_zip(a)
    else:
        print(write_status())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/phase4_lex.py
#!/usr/bin/env python3
"""CS-781 Phase 4 -- lexical baselines (TF-IDF + logistic regression). CPU only.

    python phase4_lex.py --view all  --tag lex_all
    python phase4_lex.py --view tail --chars 6000 --tag lex_tail
    python phase4_lex.py --sections          # head / middle / tail / all, one model each: the
                                             # "which sections carry the outcome" evidence

Features: hashed word 1-2-gram TF-IDF (sublinear tf, 2^21 buckets) of the chosen view,
plus hashed char 3-5-gram TF-IDF (char_wb) when --char is on. Logistic regression, C picked on
dev Macro-F1 over a small grid, balanced class weights. Writes results/<tag>/
in the shared run format so phase4_ens.py can mix it with the neural runs.

RULES NOTE. No pretrained weights and nothing outside the CJPE data, but it is
not a Phase-1 model: any submission containing it is a HYBRID, reported beside
the model-only number, never instead of it.
"""

from __future__ import annotations

import argparse
import collections
import os
import random
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from phase4_data import P4, load_all, load_jsonl, log, summarise, save_run, write_submission  # noqa: E402


def view(t, v, n):
    if v == "all":
        return t
    if v == "head":
        return t[:n]
    if v == "tail":
        return t[-n:]
    if v == "mid":
        m = len(t) // 2
        return t[max(0, m - n // 2): m + n // 2]
    raise ValueError(v)


def fit_one(data, v, chars, use_char, tag, submit, Cs, max_feat, sw=None, note=""):
    from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
    from sklearn.linear_model import LogisticRegression
    from scipy.sparse import hstack
    t0 = time.time()
    X = {s: [view(t, v, chars) for _, t, _ in data[s]] for s in data}
    y = {s: np.array([l for _, _, l in data[s]]) for s in ("train", "dev")}

    def feats(vec, tr, dv, te):
        # hashing keeps memory flat on a 12 GB Colab CPU runtime: a fitted
        # vocabulary of every word bigram in 37k judgments would not fit
        tf = TfidfTransformer(sublinear_tf=True)
        a_ = tf.fit_transform(vec.transform(tr))
        return a_, tf.transform(vec.transform(dv)), tf.transform(vec.transform(te))

    wv = HashingVectorizer(ngram_range=(1, 2), n_features=max_feat, alternate_sign=False, norm=None,
                           dtype=np.float32, lowercase=True)
    a1, b1, c1 = feats(wv, X["train"], X["dev"], X["test"])
    Mtr, Mdv, Mte = [a1], [b1], [c1]
    if use_char:
        cv = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=max_feat,
                               alternate_sign=False, norm=None, dtype=np.float32)
        tl = lambda s: [view(t, "tail", min(chars, 6000)) if v == "all" else x
                        for (_, t, _), x in zip(data[s], X[s])]
        a2, b2, c2 = feats(cv, tl("train"), tl("dev"), tl("test"))
        Mtr.append(a2)
        Mdv.append(b2)
        Mte.append(c2)
    Xtr, Xdv, Xte = hstack(Mtr).tocsr(), hstack(Mdv).tocsr(), hstack(Mte).tocsr()
    log(f"  [{tag}] view={v} chars={chars} char={use_char}: {Xtr.shape[1]} features, "
        f"vectorised in {time.time()-t0:.0f}s")
    best = None
    for C in Cs:
        clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", solver="liblinear")
        clf.fit(Xtr, y["train"], sample_weight=sw)
        zd = clf.decision_function(Xdv)
        s = summarise(y["dev"].tolist(), zd.tolist())
        log(f"    C={C:<5}: dev macro-F1 tuned {s['f1_tuned']:.4f} @0 {s['f1_at0']:.4f} AUC {s['auc']:.4f}")
        if best is None or s["f1_tuned"] > best[0]["f1_tuned"]:
            best = (s, C, zd, clf.decision_function(Xte))
    s, C, zd, zt = best
    meta = dict(tag=tag, kind="hybrid", technique="tfidf-lr", arch="lexical", view=v, chars=chars,
                char_ngrams=use_char, C=C, n_train=len(data["train"]), note=note, dev_f1=s["f1_tuned"], dev_f1_at0=s["f1_at0"], thr=s["thr"],
                dev_auc=s["auc"], dev_p=s["precision"], dev_r=s["recall"],
                n_features=int(Xtr.shape[1]), train_s=time.time() - t0, n_trainable=int(Xtr.shape[1]))
    ids_d = [r[0] for r in data["dev"]]
    ids_t = [r[0] for r in data["test"]]
    save_run(tag, ids_d, zd, ids_t, zt, meta)
    log(f"  [{tag}] BEST C={C}: dev macro-F1 {s['f1_tuned']:.4f} (thr {s['thr']:+.3f}) AUC {s['auc']:.4f} "
        f"P {s['precision']:.3f} R {s['recall']:.3f} ({time.time()-t0:.0f}s)")
    if submit:
        pred = {i: int(v_ > s["thr"]) for i, v_ in zip(ids_t, zt)}
        write_submission(tag, ids_t, pred, dev=s["f1_tuned"], kind="hybrid",
                         note=f"TF-IDF+LR view={v} chars={chars} char={use_char} n={len(data['train'])} {note}")
    return s


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--view", default="all", choices=["all", "head", "tail", "mid"])
    ap.add_argument("--chars", type=int, default=6000)
    ap.add_argument("--char", action="store_true", help="add char 3-5-gram features")
    ap.add_argument("--sections", action="store_true")
    ap.add_argument("--C", nargs="+", type=float, default=[1.0, 4.0, 16.0])
    ap.add_argument("--max-feat", type=int, default=1 << 21)
    ap.add_argument("--tag", default=None)
    ap.add_argument("--no-submit", dest="submit", action="store_false")
    ap.add_argument("--train-max", type=int, default=0, help="use only this many train docs (seed-781 order)")
    ap.add_argument("--extra", default=None, help="external jsonl (phase4_external.py output)")
    ap.add_argument("--extra-max", type=int, default=0)
    ap.add_argument("--extra-weight", type=float, default=1.0)
    ap.add_argument("--curve", action="store_true",
                    help="learning curve: dev F1 at 2.5k/5k/10k/20k/all gold docs (+ all + extra)")
    a = ap.parse_args()
    data = load_all()
    gold = data["train"]
    perm = list(range(len(gold)))
    random.Random(781).shuffle(perm)            # same order as phase4_cjpe.py
    ext = load_jsonl(a.extra, a.extra_max or None) if a.extra else []
    if a.extra:
        log(f"  extra: {len(ext)} docs from {a.extra}, labels {dict(collections.Counter(l for _, _, l in ext))}, "
            f"weight {a.extra_weight}")

    def with_train(n, use_ext):
        d = dict(data)
        tr = [gold[i] for i in perm[:n]] if n else list(gold)
        w = [1.0] * len(tr)
        if use_ext:
            tr += ext
            w += [a.extra_weight] * len(ext)
        d["train"] = tr
        return d, np.array(w)

    if a.sections:
        out = {}
        for v in ("head", "mid", "tail", "all"):
            out[v] = fit_one(data, v, a.chars, False, f"lexsec_{v}", False, a.C, a.max_feat)
        log("\n  section evidence (word TF-IDF, same model, different part of the judgment):")
        for v, s in out.items():
            log(f"    {v:5s} {a.chars if v != 'all' else 'all'} chars: dev macro-F1 {s['f1_tuned']:.4f} AUC {s['auc']:.4f}")
        return
    if a.curve:
        rows = []
        sizes = [n for n in (2500, 5000, 10000, 20000) if n < len(gold)] + [0]
        for n in sizes:
            d, w = with_train(n, False)
            s = fit_one(d, a.view, a.chars, False, f"lexcurve_{n or 'all'}", False, a.C, a.max_feat, w)
            rows.append((len(d["train"]), "gold", s))
        if ext:
            d, w = with_train(0, True)
            s = fit_one(d, a.view, a.chars, False, "lexcurve_all_ext", False, a.C, a.max_feat, w)
            rows.append((len(d["train"]), f"gold + {len(ext)} silver", s))
        log(f"\n  learning curve (word TF-IDF, view={a.view}):")
        for n, what, s in rows:
            log(f"    {n:6d} docs ({what}): dev macro-F1 {s['f1_tuned']:.4f} AUC {s['auc']:.4f}")
        return
    d, w = with_train(a.train_max, bool(ext))
    tag = a.tag or f"lex_{a.view}"
    fit_one(d, a.view, a.chars, a.char, tag, a.submit, a.C, a.max_feat, w,
            note=f"+{len(ext)} silver" if ext else "")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/phase4_overlap.py
#!/usr/bin/env python3
"""CS-781 Phase 4 -- integrity check: do CJPE test judgments sit in the Phase-1 corpus?

    python phase4_overlap.py                       # Supreme Court corpus (1.3 GB), CPU, ~10-20 min
    python phase4_overlap.py --sources train_supreme_court train_high_courts

WHY. CJPE text is a judgment with its final order cut off. The Phase-1
pretraining corpus holds full Supreme Court judgments. If a test case is in
there, the corpus copy carries the sentence that says "the appeal is allowed",
and predicting from it is reading the answer, not predicting. Phase 1 used
corpus overlap openly as a hybrid; for outcome prediction that defence is much
weaker. THIS SCRIPT ONLY MEASURES. It writes no submission. Using a match in a
submission needs Nithin's explicit go-ahead after asking the TAs.

METHOD. Each CJPE document gives up to 10 needles of 12 normalised words,
evenly spaced over the document, each starting on a word of >= 7 letters so the
corpus scan can skip most positions. A corpus record that contains >= 3 needles
of one document is a match. For matched dev documents the corpus text after the
last needle is searched for an outcome phrase, and that rule's agreement with
the gold label says how much a leak would be worth.
"""

from __future__ import annotations

import argparse
import collections
import json
import os
import re
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from phase4_data import P4, load_split, log, atomic_json  # noqa: E402

WORD = re.compile(r"[a-z0-9]+")
N = 12
ALLOW = re.compile(r"(appeals?|petitions?) (is|are|stand|stands) (hereby )?(allowed|accepted)|"
                   r"we (accordingly |therefore )?allow (the|this|these)|set aside the (impugned )?(judgment|order)", re.I)
DISMISS = re.compile(r"(appeals?|petitions?) (is|are|stand|stands) (hereby )?(dismissed|rejected)|"
                     r"we (accordingly |therefore )?dismiss (the|this|these)|no merit in (the|this|these)", re.I)


def stream_records(path, bufsize=1 << 22):
    """Every record of a large JSON array without loading the file (as in Phase 1)."""
    dec = json.JSONDecoder()
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        buf = f.read(bufsize)
        pos = buf.find("{")
        if pos < 0:
            return
        while True:
            try:
                obj, end = dec.raw_decode(buf, pos)
            except ValueError:
                more = f.read(bufsize)
                if not more:
                    return
                buf = buf[pos:] + more
                pos = buf.find("{")
                if pos < 0:
                    return
                continue
            yield obj
            k = end
            while k < len(buf) and buf[k] in ", \n\r\t":
                k += 1
            if k >= len(buf) or buf[k] == "]":
                more = f.read(bufsize)
                if not more:
                    return
                buf = buf[k:] + more
                pos = buf.find("{")
                if pos < 0:
                    return
            elif len(buf) - k < (1 << 20):
                more = f.read(bufsize)
                buf = buf[k:] + (more or "")
                pos = buf.find("{")
                if pos < 0:
                    return
            else:
                pos = k


def needles(text, k=10):
    w = WORD.findall(text.lower())
    out = []
    if len(w) < N + 5:
        return out
    for j in range(k):
        i = int((j + 0.5) * (len(w) - N) / k)
        while i < len(w) - N and len(w[i]) < 7:
            i += 1
        if i < len(w) - N:
            out.append((" ".join(w[i:i + N]), j))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sources", nargs="+", default=["train_supreme_court"])
    ap.add_argument("--min-hits", type=int, default=3)
    ap.add_argument("--corpus-path", default=None, help="local corpus JSON instead of the HF download")
    a = ap.parse_args()
    from huggingface_hub import hf_hub_download

    docs = {}
    for split in ("test", "dev"):
        for d, t, l in load_split(split):
            docs[(split, d)] = (t, l)
    index = collections.defaultdict(list)
    first = set()
    for key, (t, _) in docs.items():
        for s, j in needles(t):
            index[hash(s)].append((key, j))
            first.add(s.split(" ", 1)[0])
    log(f"  {len(docs)} CJPE docs (test + dev), {len(index)} needles, {len(first)} distinct first words")

    found = {}                                   # key -> (source, record no, hits, tail text)
    for src in a.sources:
        path = a.corpus_path or hf_hub_download("Exploration-Lab/CS-781-Capstone",
                                                f"Phase_1_Pretraining/Train/{src}.json", repo_type="dataset")
        t0 = time.time()
        for n, rec in enumerate(stream_records(path)):
            text = rec.get("text") or ""
            w = WORD.findall(text.lower())
            hits = collections.defaultdict(set)
            last = {}
            for i in range(len(w) - N + 1):
                if w[i] not in first:
                    continue
                h = hash(" ".join(w[i:i + N]))
                for key, j in index.get(h, ()):
                    hits[key].add(j)
                    last[key] = i + N
            for key, js in hits.items():
                if len(js) >= a.min_hits and (key not in found or len(js) > found[key][2]):
                    after = " ".join(w[last[key]:])
                    found[key] = (src, n, len(js), after[-3000:], len(w) - last[key])
            if n % 20000 == 0 and n:
                log(f"    {src}: {n} records, {len(found)} matched so far ({time.time()-t0:.0f}s)")
        log(f"  {src}: done, {n+1} records in {time.time()-t0:.0f}s")

    res = {}
    for split in ("test", "dev"):
        keys = [k for k in docs if k[0] == split]
        m = [k for k in keys if k in found]
        extra = sorted(found[k][4] for k in m)
        res[split] = dict(n=len(keys), matched=len(m),
                          median_words_after_last_needle=extra[len(extra) // 2] if extra else 0)
        log(f"\n  {split}: {len(m)} of {len(keys)} documents ({100*len(m)/max(1,len(keys)):.1f}%) found in "
            f"{a.sources}; median {res[split]['median_words_after_last_needle']} corpus words after the "
            f"last matched needle")
    # what would the leak be worth? outcome phrases in the corpus text that follows, on DEV
    agree = tot = cov = 0
    for k in docs:
        if k[0] != "dev" or k not in found:
            continue
        tail = found[k][3]
        al, di = bool(ALLOW.search(tail)), bool(DISMISS.search(tail))
        tot += 1
        if al != di:
            cov += 1
            agree += int((1 if al else 0) == docs[k][1])
    if tot:
        log(f"  dev leak value: of {tot} matched dev docs, the corpus continuation states an outcome for "
            f"{cov}; that rule agrees with the gold label on {agree}/{cov} ({100*agree/max(1,cov):.1f}%)")
    res["dev_rule"] = dict(matched=tot, decided=cov, agree=agree)
    res["sources"] = a.sources
    os.makedirs(P4, exist_ok=True)
    atomic_json(res, f"{P4}/integrity.json", indent=1)
    log(f"  wrote {P4}/integrity.json -- MEASUREMENT ONLY, nothing submitted")
    if res["test"]["matched"] > 0.05 * res["test"]["n"]:
        log("  >>> LARGE OVERLAP. Stop and ask the TAs before using any of it. <<<")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/cs781_capstone/phase4/code/run_p4.sh
#!/bin/bash
# Phase 4 GPU queue, ONE job at a time on the L4 (two jobs split the GPU, three
# OOM'd in Phase 3). Ordered by what the grade and the leaderboard need first,
# so if compute runs out part-way, what finished is the most valuable part.
# Re-running is safe: a run whose meta.json exists is skipped.
#
#   0. bench        30 updates each of M4 full / M4 LoRMA / M2 full: measured tokens/s
#                   -> projected minutes for every run printed before it starts
#   1. m4_lorma     LoRMA on M4 (COMPULSORY) + KD-pool predictions (teacher candidate)
#   2. m4_full      full fine-tuning of M4 + KD-pool predictions (teacher candidate)
#   3. kd_m4half    half-depth M4 student distilled from ens(m4_full, m4_lorma) on
#                   8k held-out train docs + the test text (COMPULSORY KD)
#   4. m1_adapter   Houlsby adapters on M1
#   5. m3_adalora   AdaLoRA on M3
#   6. m2_unfreeze  gradual unfreezing of the M2 encoder
#   7. m1_bitfit    BitFit on M1
#   8. m2_partial   top-6 blocks of M2 + layer-wise LR decay
#   9. ensembles -> zips (CPU work, seconds)
#  10. m4_rest      better M4 method on ALL gold docs except the KD pool (~29k docs)
#  11. m4_silver    same on 10k gold + 15k silver docs from the Phase-1 corpus (if built)
#  12. final ensembles
#
# Knobs (environment): P4_DOCS (train docs per run, default 10000),
# P4_ARGS (extra args for every run), P4_ONLY (space-separated tags to run).

cd "$(dirname "$0")"
P4=${CS781_P4:-/content/drive/MyDrive/cs781_capstone/phase4}
L=$P4/logs
mkdir -p "$L"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True OMP_NUM_THREADS=$(nproc)
DOCS=${P4_DOCS:-10000}
COMMON="--train-docs $DOCS --kd-pool 8000 --chunks 2 --chunk 512 --batch 8 --accum 2 --evals 3 --submit --save-ckpt ${P4_ARGS:-}"
Q=$L/queue.txt
say() { echo "[$(date +%H:%M)] $*" | tee -a "$Q"; }

project() {   # project <bench tag>: minutes for one run at the measured speed
  python - "$P4/bench/$1.json" "$DOCS" <<'EOF'
import json, sys, os
p, docs = sys.argv[1], int(sys.argv[2])
if os.path.exists(p):
    b = json.load(open(p))
    m = docs * 1024 / b["tok_per_s"] / 60
    print(f"{b['tok_per_s']/1e3:.1f}k tok/s, peak {b['peak_gb']:.1f} GB -> ~{m:.0f} min training + ~5 min eval")
else:
    print("no bench")
EOF
}

run() {
  local tag=$1; shift
  if [ -n "${P4_ONLY:-}" ] && [[ " $P4_ONLY " != *" $tag "* ]]; then return; fi
  if [ -f "$P4/results/$tag/meta.json" ]; then say "skip $tag (done)"; return; fi
  local extra=""
  for attempt in 1 2 3; do
    say "start $tag $extra"
    local n0=$(cat "$L/q_$tag.out" 2>/dev/null | wc -l)
    python phase4_cjpe.py --tag "$tag" $COMMON "$@" $extra >> "$L/q_$tag.out" 2>&1
    if [ -f "$P4/results/$tag/meta.json" ]; then
      say "done  $tag | $(grep -a '=====.*dev macro-F1' "$L/q_$tag.out" | tail -1 | sed 's/^=* *//')"
      return
    fi
    if tail -n +$((n0 + 1)) "$L/q_$tag.out" | grep -aq "OutOfMemoryError\|CUDA out of memory"; then
      say "OOM   $tag (attempt $attempt) -- retrying with --grad-ckpt"
      extra="--grad-ckpt"; sleep 20
    else
      say "FAIL  $tag -- see logs/q_$tag.out"; tail -n 5 "$L/q_$tag.out" | tee -a "$Q"; return
    fi
  done
}

bench() {
  local tag=$1; shift
  [ -f "$P4/bench/$tag.json" ] && return
  python phase4_cjpe.py --tag "$tag" $COMMON --bench --max-steps 30 "$@" >> "$L/q_bench.out" 2>&1 \
    || python phase4_cjpe.py --tag "$tag" $COMMON --bench --max-steps 30 --grad-ckpt "$@" >> "$L/q_bench.out" 2>&1
  say "bench $tag: $(project $tag)"
}

ens() {   # ens <tag> <args>
  local tag=$1; shift
  python phase4_ens.py ens --tag "$tag" "$@" > "$L/$tag.out" 2>&1
  grep -a "dev macro-F1\|WROTE\|teacher\|Error\|Traceback" "$L/$tag.out" | tee -a "$Q"
}

say "queue: $(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null) | $DOCS docs/run"
python phase4_data.py tokenize >> "$L/q_tokenize.out" 2>&1 || { say "tokenize FAILED"; tail -n 5 "$L/q_tokenize.out"; exit 1; }

bench bench_m4_full  --model M4_LexArcP --method full
bench bench_m4_lorma --model M4_LexArcP --method lorma
bench bench_m2_full  --model M2_LexArcED --method full

run m4_lorma    --model M4_LexArcP  --method lorma --rank 16 --dump-kd
run m4_full     --model M4_LexArcP  --method full --dump-kd
if [ ! -f "$P4/teacher_ens_T1.pt" ]; then
  ens ens_T1 --runs m4_full m4_lorma --teacher
fi
if [ -f "$P4/teacher_ens_T1.pt" ]; then
  run kd_m4half --model M4_LexArcP --method full --keep-every 2 --kd "$P4/teacher_ens_T1.pt" --kd-test
fi
run m1_adapter  --model M1_SC       --method adapter --adapter-dim 64
run m3_adalora  --model M3_LexArcL  --method adalora --rank 16 --ada-target 6
run m2_unfreeze --model M2_LexArcED --method unfreeze --llrd 0.9
run m1_bitfit   --model M1_SC       --method bitfit
run m2_partial  --model M2_LexArcED --method partial --top-k 6 --llrd 0.9

ens ens_model_all --kind model
ens ens_model_greedy --kind model --greedy 8
ens ens_hyb_greedy --greedy 8
ens ens_model_rate --kind model --greedy 8 --rate
# ---- top-score extras (after every required run): more data on the better M4 method
BEST=$(python - "$P4" <<'EOF2'
import json, os, sys
p = sys.argv[1]
f = lambda t: json.load(open(f"{p}/results/{t}/meta.json"))["dev_f1"] if os.path.exists(f"{p}/results/{t}/meta.json") else -1
print("lorma --rank 16" if f("m4_lorma") > f("m4_full") else "full")
EOF2
)
say "extras use M4 method: $BEST"
run m4_rest --model M4_LexArcP --method $BEST --use-rest          # all gold except the KD pool (~3x data)
SILVER=$P4/external/sc_silver.jsonl
if [ -f "$SILVER" ]; then
  run m4_silver --model M4_LexArcP --method $BEST --extra "$SILVER" --extra-docs 15000 --extra-weight 0.5
else
  say "skip m4_silver: $SILVER not built (run phase4_external.py)"
fi
ens ens_final_model --kind model --greedy 8
ens ens_final_hyb --greedy 8
ens ens_final_rate --kind model --greedy 8 --rate
python phase4_ens.py status > /dev/null 2>&1
say "QUEUE DONE"

In [ ]:
# 3. helpers (both runtimes)
os.chdir(CODE)
def sh(cmd):
    print('$', cmd); r = subprocess.run(cmd, shell=True, cwd=CODE); return r.returncode
def status():
    q = f'{P4}/logs/queue.txt'
    if os.path.exists(q):
        print(''.join(open(q).readlines()[-25:]))
    for p in sorted(glob.glob(f'{P4}/logs/live_*.txt'), key=os.path.getmtime)[-2:]:
        print(os.path.basename(p), '|', ''.join(open(p).readlines()[-3:]))
    sh('python phase4_ens.py status | head -40')
print('ok, cwd', os.getcwd())

## A. CPU runtime
A1 data gate · A2 integrity check (Phase-1 Supreme Court corpus vs test) · A3 TF-IDF zips · A4 section evidence.
Read the A1 and A2 output before trusting any model number.

In [ ]:
# A1. data gate: sizes, label balance, lengths, leftover outcome sentences, train/dev/test duplicates
sh(f'python phase4_data.py gate 2>&1 | tee {P4}/logs/00_gate.log')

In [ ]:
# A2. integrity: are test judgments (with their final orders) in the Phase-1 pretraining corpus?
# MEASUREMENT ONLY -- writes phase4/integrity.json, never a submission. Downloads 1.3 GB. ~10-20 min.
sh(f'python phase4_overlap.py 2>&1 | tee {P4}/logs/01_integrity.log')

In [ ]:
# A3. TF-IDF + logistic regression -> hybrid zips lex_all, lex_tail (~10-20 min on 2 CPUs)
sh(f'python phase4_lex.py --view all --tag lex_all 2>&1 | tee {P4}/logs/lex_all.log')
sh(f'python phase4_lex.py --view tail --chars 6000 --char --tag lex_tail 2>&1 | tee {P4}/logs/lex_tail.log')

In [ ]:
# A4. which part of the judgment carries the outcome? same model on head / middle / tail / all (report §10.5)
sh(f'python phase4_lex.py --sections 2>&1 | tee {P4}/logs/lex_sections.log')

## B. GPU runtime (L4)
B1 starts the whole queue in the background and returns at once; B2 shows progress (re-run it any time).
The queue tokenises first (CPU, a few minutes), benchmarks speed, then runs in priority order:
LoRMA (compulsory) → full FT → KD student (compulsory) → adapters → AdaLoRA → gradual unfreezing → BitFit → partial FT,
then ensembles. If compute runs low, stop after `kd_m4half` + `m1_adapter` + `m3_adalora`: that already covers
6 techniques and 3 architectures; `m2_unfreeze` adds the 4th architecture.

In [ ]:
# B1. launch the queue (safe to re-run: finished runs are skipped)
import torch
assert torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8, 'need an L4/A100, not a T4'
os.environ.setdefault('P4_DOCS', '10000')       # train docs per run
log = open(f'{P4}/logs/queue_stdout.txt', 'a')
proc = subprocess.Popen(['bash', 'run_p4.sh'], cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                        start_new_session=True)
print('queue started, pid', proc.pid)

In [ ]:
# B2. progress (re-run any time; also readable from any runtime / phone: phase4/logs/queue.txt)
status()

In [ ]:
# B3. (optional, after the queue) list the zips ready to upload, best dev first
for t in sorted(json.load(open(f'{P4}/submissions/tracker.json')), key=lambda t: -(t.get('dev') or 0)):
    print(f"{t['zip']:32s} {t['kind']:6s} dev {t.get('dev') or 0:.4f}  test pos-rate {t['pos_rate']:.3f}  {t['note']}")